# 本地图片（单张/文件夹）→ 4 子图 → VGGT 批量重建 GLB

这个 notebook 用于：
1. 支持 `input_path` 为单张图片或图片文件夹
2. 每张图按 512×512 处理并裁成 4 张 256×256 子图
3. 用 VGGT 推理并生成 `.glb`
4. 结果按原图文件名对应保存到 `output_path`

## 1) 安装与导入依赖（PyTorch / OpenCV / NumPy / Trimesh）

In [1]:
# 如环境已安装可注释掉本单元
# %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# %pip install -q opencv-python numpy trimesh ipywidgets

import os
import sys
import glob
import gc
import time
import tempfile
import json
import re
from pathlib import Path

import cv2
import torch
import numpy as np
import trimesh

from IPython.display import display, HTML, FileLink
import matplotlib.pyplot as plt

# 关键：保证可导入 vggt 代码
# 如果你的 notebook 不在 /home/viplab/liukejia/project/vggt 下，请改这里
PROJECT_ROOT = Path('/home/viplab/liukejia/project/vggt').resolve()
os.chdir(PROJECT_ROOT)
sys.path.append(str(PROJECT_ROOT))
sys.path.append(str(PROJECT_ROOT / 'vggt'))
sys.path.append(str(PROJECT_ROOT.parent / 'reconstruct'))

from visual_util import predictions_to_glb
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map
from utils.camera import parse_json_camera_extrinsic_intrinsic

print('PROJECT_ROOT =', PROJECT_ROOT)

/home/viplab/anaconda3/envs/vggt/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT = /home/viplab/liukejia/project/vggt


## 0) 参数配置（最前设置）

In [ ]:
# ===== 必填参数 =====
# 可填单张图片路径，或文件夹路径（会处理文件夹内所有图片）
# input_path = '/home/viplab/liukejia/project/mv_model/results_4in1_sdxl_warp_ckpt13000'
# input_path = '/home/viplab/liukejia/project/mv_model/results_4in1_sdxl_warp_ckpt_21500'
input_path = '/home/viplab/liukejia/project/result/render_3d_stitched/'
# 输出目录：每张图会生成同名 .glb（以及可选 .npz）
output_path = '/home/viplab/liukejia/project/result/reconstruct_pred_scale2/'
# 深度输出目录：按你的要求输出到 {input_path}_depth
depth_output_path = f"{input_path.rstrip('/')}_depth"

# ===== 处理参数 =====
force_resize = True  # 非 512x512 是否强制缩放到 512x512
conf_thres = 20.0
prediction_mode = 'Depthmap and Camera Branch'
save_npz = False  # 批量时建议关闭，避免写大 npz 太慢/中断

# ===== 输出开关 =====
only_output_depth = False  # True: 仅输出深度，不导出 GLB/NPZ

# ===== 深度导出参数（仅原始深度） =====
save_raw_depth_npy = False  # True: 输出原始深度图的 .npy；False: 不输出
if only_output_depth and not save_raw_depth_npy:
    print('[Warn] only_output_depth=True 但 save_raw_depth_npy=False，已自动开启深度导出')
    save_raw_depth_npy = True

# ===== 相机参数来源配置 =====
# True: 使用 camconf 外部相机；False: 使用 VGGT 预测相机
use_camconf_camera = True
camconf_dir = '/data/test_grid4_from_test_grid4/camconf'
# 仅使用该元组第一个角度（默认 0 度）作为锚点相机
camconf_angle_order = (0, 90, 180, 270)
# True: 缺少/异常 camconf 直接报错；False: 回退到 prediction
strict_camconf = True
# 注意：开启会改变每个相机自身朝向，破坏“仅全局刚体变换”的几何不变性
invert_camera_direction = False
# 打印相机调试信息：变换前 / 变换后 / 文件中
debug_print_camera = True

# ===== 只做刚体/相似变换（保几何） =====
# none | rigid | similarity | camera_fit
post_align_mode = 'camera_fit'
post_align_scale = 1.0  # similarity 模式有效；rigid/camera_fit 下通常保持 1.0

# ===== camera_fit 额外缩放：让预测点范围贴近文件相机范围 =====
camera_fit_match_range = True
camera_fit_range_gain = 1.0    # 1.0=完全按范围比例缩放，0.5=只走一半
camera_fit_scale_min = 0.05    # 防止缩放过小
camera_fit_scale_max = 20.0    # 防止缩放过大
# True: 范围缩放同时作用“点云+相机”；False: 仅作用点云
camera_fit_range_apply_to_camera = True

# 支持的图片后缀
valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

print('input_path =', input_path)
print('output_path =', output_path)
print('depth_output_path =', depth_output_path)
print('only_output_depth =', only_output_depth)
print('save_raw_depth_npy =', save_raw_depth_npy)
print('use_camconf_camera =', use_camconf_camera)
print('camconf_dir =', camconf_dir)
print('camconf_angle_order =', camconf_angle_order)
print('strict_camconf =', strict_camconf)
print('invert_camera_direction =', invert_camera_direction)
print('debug_print_camera =', debug_print_camera)
print('post_align_mode =', post_align_mode)
print('post_align_scale =', post_align_scale)
print('camera_fit_match_range =', camera_fit_match_range)
print('camera_fit_range_gain =', camera_fit_range_gain)
print('camera_fit_scale_min/max =', (camera_fit_scale_min, camera_fit_scale_max))
print('camera_fit_range_apply_to_camera =', camera_fit_range_apply_to_camera)
print('save_npz =', save_npz)

input_path = /home/viplab/liukejia/project/result/render_3d_stitched/
output_path = /home/viplab/liukejia/project/result/reconstruct_pred_scale2/
depth_output_path = /home/viplab/liukejia/project/result/render_3d_stitched_depth
only_output_depth = False
save_raw_depth_npy = False
use_camconf_camera = True
camconf_dir = /data/test_grid4_from_test_grid4/camconf
camconf_angle_order = (0, 90, 180, 270)
strict_camconf = True
invert_camera_direction = False
debug_print_camera = True
post_align_mode = camera_fit
post_align_scale = 1.0
camera_fit_match_range = True
camera_fit_range_gain = 1.0
camera_fit_scale_min/max = (0.05, 200.0)
camera_fit_range_apply_to_camera = True
save_npz = False


## 2) 加载 VGGT 模型与权重到 CUDA

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device)

print('Initializing VGGT...')
model = VGGT()
_URL = 'https://huggingface.co/facebook/VGGT-1B/resolve/main/model.pt'
state_dict = torch.hub.load_state_dict_from_url(_URL, map_location='cpu')
model.load_state_dict(state_dict)
model.eval().to(device)

# 半精度配置（仅 GPU 有意义）
if torch.cuda.is_available():
    major_cc = torch.cuda.get_device_capability()[0]
    amp_dtype = torch.bfloat16 if major_cc >= 8 else torch.float16
else:
    amp_dtype = None

print('amp_dtype =', amp_dtype)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

device = cuda
Initializing VGGT...


amp_dtype = torch.bfloat16


## 3) 读取 `input_path`（单图或文件夹）并做尺寸校验

In [4]:
def collect_input_images(input_path: str, valid_exts=None):
    if valid_exts is None:
        valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

    p = Path(input_path)
    if not p.exists():
        raise FileNotFoundError(f'input_path not found: {p}')

    if p.is_file():
        if p.suffix.lower() not in valid_exts:
            raise ValueError(f'Unsupported file extension: {p.suffix}')
        return [p]

    if p.is_dir():
        files = [x for x in sorted(p.iterdir()) if x.is_file() and x.suffix.lower() in valid_exts]
        if len(files) == 0:
            raise ValueError(f'No image files found in folder: {p}')
        return files

    raise ValueError(f'Invalid input_path: {p}')


def read_single_image_512(image_path: str, force_resize: bool = True):
    image_path = Path(image_path)
    bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError(f'Failed to read image: {image_path}')

    h, w = bgr.shape[:2]
    if (h, w) != (512, 512):
        if force_resize:
            bgr = cv2.resize(bgr, (512, 512), interpolation=cv2.INTER_AREA)
            print(f'[Info] resized {image_path.name}: {(w, h)} -> (512, 512)')
        else:
            raise ValueError(f'{image_path.name}: expect 512x512, got {(w, h)}')

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return rgb

input_images = collect_input_images(input_path, valid_exts=valid_exts)
print(f'Found {len(input_images)} image(s).')
print('First 5:', [p.name for p in input_images[:5]])

Found 9 image(s).
First 5: ['num123_-800_100_0_0007.png', 'num123_-800_150_0_0008.png', 'num123_-800_200_0_0009.png', 'num123_-850_100_0_0004.png', 'num123_-850_150_0_0005.png']


## 4) 将单张图裁成 4 个 256×256 子图并写入 `target_dir/images`

In [5]:
def split_512_to_4x256_and_save(img_rgb: np.ndarray, target_dir: str):
    target_dir = Path(target_dir)
    images_dir = target_dir / 'images'
    images_dir.mkdir(parents=True, exist_ok=True)

    # 固定顺序：左上、右上、左下、右下
    patches = [
        img_rgb[0:256, 0:256],      # 000000
        img_rgb[0:256, 256:512],    # 000001
        img_rgb[256:512, 0:256],    # 000002
        img_rgb[256:512, 256:512],  # 000003
    ]

    patch_paths = []
    for i, p in enumerate(patches):
        out_path = images_dir / f'{i:06d}.png'
        cv2.imwrite(str(out_path), cv2.cvtColor(p, cv2.COLOR_RGB2BGR))
        patch_paths.append(str(out_path))

    cv2.imwrite(str(target_dir / 'original_512.png'), cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
    return patch_paths

# 仅快速检查第一张（可选）
_preview_rgb = read_single_image_512(str(input_images[0]), force_resize=force_resize)
_preview_dir = Path(tempfile.mkdtemp(prefix='vggt_preview_'))
_ = split_512_to_4x256_and_save(_preview_rgb, _preview_dir)
print('preview target_dir =', _preview_dir)

preview target_dir = /tmp/vggt_preview_n5d_2296


## 5) 运行 VGGT 推理并生成 `extrinsic/intrinsic/world_points`

In [6]:
def extract_camconf_base_stem(image_stem: str) -> str:
    """
    例：num123_10_0_0_0011 -> 10_0_0
    """
    m = re.match(r'^num123_([^_]+_[^_]+_[^_]+)_[^_]+$', image_stem)
    if m:
        return m.group(1)

    parts = image_stem.split('_')
    if len(parts) >= 5 and parts[0] == 'num123':
        return '_'.join(parts[1:4])
    if len(parts) >= 3:
        return '_'.join(parts[:3])

    raise ValueError(f'Cannot parse camconf stem from image stem: {image_stem}')


def load_camconf_anchor_camera_for_image(
    image_path: Path,
    camconf_dir: str,
    anchor_angle: int = 0,
    H: int = 256,
    W: int = 256,
):
    """
    只读取 0 度（或指定角度）camconf，作为锚点相机。
    """
    camconf_dir = Path(camconf_dir)
    if not camconf_dir.exists():
        raise FileNotFoundError(f'camconf_dir not found: {camconf_dir}')

    base_stem = extract_camconf_base_stem(image_path.stem)
    json_path = camconf_dir / f'{base_stem}_{anchor_angle}_camconf.json'
    if not json_path.exists():
        raise FileNotFoundError(f'camconf file missing: {json_path}')

    ext44, intr33 = parse_json_camera_extrinsic_intrinsic(
        str(json_path), device='cpu', H=H, W=W, dtype=torch.float32, flip_y=True
    )
    return (
        ext44.squeeze(0).detach().cpu().numpy().astype(np.float32),
        intr33.squeeze(0).detach().cpu().numpy().astype(np.float32),
        str(json_path),
    )


def _convert_extrinsic_shape_like(src_extr: np.ndarray, ref_extr: np.ndarray) -> np.ndarray:
    src = np.asarray(src_extr, dtype=np.float32)
    ref = np.asarray(ref_extr)

    if src.ndim != 3 or ref.ndim != 3:
        raise ValueError(f'extrinsic ndim error: src={src.shape}, ref={ref.shape}')

    if ref.shape[1:] == (3, 4):
        if src.shape[1:] == (3, 4):
            return src
        if src.shape[1:] == (4, 4):
            return src[:, :3, :4]
        raise ValueError(f'unsupported src extrinsic shape: {src.shape}')

    if ref.shape[1:] == (4, 4):
        if src.shape[1:] == (4, 4):
            return src
        if src.shape[1:] == (3, 4):
            out = np.tile(np.eye(4, dtype=np.float32), (src.shape[0], 1, 1))
            out[:, :3, :4] = src
            return out
        raise ValueError(f'unsupported src extrinsic shape: {src.shape}')

    raise ValueError(f'unsupported ref extrinsic shape: {ref.shape}')


def _flip_camera_view_direction(extrinsic: np.ndarray) -> np.ndarray:
    ext = np.asarray(extrinsic, dtype=np.float32)
    if ext.ndim != 3:
        raise ValueError(f'extrinsic ndim error: {ext.shape}')

    if ext.shape[1:] == (3, 4):
        ext44 = np.tile(np.eye(4, dtype=np.float32), (ext.shape[0], 1, 1))
        ext44[:, :3, :4] = ext
        return34 = True
    elif ext.shape[1:] == (4, 4):
        ext44 = ext.copy()
        return34 = False
    else:
        raise ValueError(f'unsupported extrinsic shape: {ext.shape}')

    c2w = np.linalg.inv(ext44)
    R_flip = np.diag([-1.0, 1.0, -1.0]).astype(np.float32)
    c2w[:, :3, :3] = np.einsum('nij,jk->nik', c2w[:, :3, :3], R_flip)
    ext_new = np.linalg.inv(c2w).astype(np.float32)

    if return34:
        return ext_new[:, :3, :4]
    return ext_new


def _rigid_align_all_cameras_by_cam0(pred_extr: np.ndarray, anchor_ext44: np.ndarray) -> np.ndarray:
    """
    用第 0 个预测相机对齐到锚点相机（camconf 0 度），
    再把同一刚体变换应用到所有预测相机。
    """
    pred_extr = np.asarray(pred_extr, dtype=np.float32)
    pred_extr4 = _convert_extrinsic_shape_like(
        pred_extr, np.zeros((pred_extr.shape[0], 4, 4), dtype=np.float32)
    )

    pred_c2w = np.linalg.inv(pred_extr4)
    anchor_c2w = np.linalg.inv(np.asarray(anchor_ext44, dtype=np.float32))

    R_pred0 = pred_c2w[0, :3, :3]
    C_pred0 = pred_c2w[0, :3, 3]
    R_anc0 = anchor_c2w[:3, :3]
    C_anc0 = anchor_c2w[:3, 3]

    Q = R_anc0 @ R_pred0.T
    d = C_anc0 - Q @ C_pred0

    pred_c2w_new = pred_c2w.copy()
    pred_c2w_new[:, :3, :3] = np.einsum('ij,njk->nik', Q, pred_c2w[:, :3, :3])
    pred_c2w_new[:, :3, 3] = np.einsum('ij,nj->ni', Q, pred_c2w[:, :3, 3]) + d

    pred_extr_new4 = np.linalg.inv(pred_c2w_new).astype(np.float32)
    return _convert_extrinsic_shape_like(pred_extr_new4, pred_extr)


def _camera_centers_from_extr(extrinsic: np.ndarray) -> np.ndarray:
    ext = np.asarray(extrinsic, dtype=np.float32)
    ext4 = _convert_extrinsic_shape_like(ext, np.zeros((ext.shape[0], 4, 4), dtype=np.float32))
    c2w = np.linalg.inv(ext4)
    return c2w[:, :3, 3]


def _rotation_matrix_to_euler_xyz_deg(R: np.ndarray) -> np.ndarray:
    """
    单个 3x3 旋转矩阵 -> XYZ 欧拉角（度）。
    """
    sy = np.sqrt(R[0, 0] * R[0, 0] + R[1, 0] * R[1, 0])
    singular = sy < 1e-6

    if not singular:
        x = np.arctan2(R[2, 1], R[2, 2])
        y = np.arctan2(-R[2, 0], sy)
        z = np.arctan2(R[1, 0], R[0, 0])
    else:
        x = np.arctan2(-R[1, 2], R[1, 1])
        y = np.arctan2(-R[2, 0], sy)
        z = 0.0

    return np.degrees(np.array([x, y, z], dtype=np.float32))


def _camera_euler_xyz_deg_from_extr(extrinsic: np.ndarray) -> np.ndarray:
    """
    批量外参 -> 相机 c2w 的 XYZ 欧拉角（度）。
    """
    ext = np.asarray(extrinsic, dtype=np.float32)
    ext4 = _convert_extrinsic_shape_like(ext, np.zeros((ext.shape[0], 4, 4), dtype=np.float32))
    c2w = np.linalg.inv(ext4)
    eulers = []
    for i in range(c2w.shape[0]):
        eulers.append(_rotation_matrix_to_euler_xyz_deg(c2w[i, :3, :3]))
    return np.stack(eulers, axis=0).astype(np.float32)
def _camera_forward_vecs_from_extr(extrinsic: np.ndarray) -> np.ndarray:
    """
    批量外参 -> 相机朝前方向（世界坐标系下，按 -Z 作为 forward）。
    """
    ext = np.asarray(extrinsic, dtype=np.float32)
    ext4 = _convert_extrinsic_shape_like(ext, np.zeros((ext.shape[0], 4, 4), dtype=np.float32))
    c2w = np.linalg.inv(ext4)
    forward = -c2w[:, :3, 2]
    norm = np.linalg.norm(forward, axis=1, keepdims=True) + 1e-8
    return (forward / norm).astype(np.float32)
def _camera_forward_angle_to_point_center(extrinsic: np.ndarray, world_points: np.ndarray):
    """
    返回：点云中心、相机前向向量、相机到点云中心向量、夹角（度）。
    """
    centers = _camera_centers_from_extr(extrinsic)
    forward = _camera_forward_vecs_from_extr(extrinsic)
    pts = np.asarray(world_points, dtype=np.float32).reshape(-1, 3)
    point_center = pts.mean(axis=0)
    to_center = point_center[None, :] - centers
    to_center_norm = np.linalg.norm(to_center, axis=1, keepdims=True) + 1e-8
    to_center_unit = to_center / to_center_norm
    cosv = np.sum(forward * to_center_unit, axis=1)
    cosv = np.clip(cosv, -1.0, 1.0)
    angles = np.degrees(np.arccos(cosv)).astype(np.float32)
    return point_center.astype(np.float32), forward.astype(np.float32), to_center.astype(np.float32), angles


def run_model(
    target_dir: str,
    model,
    image_path: Path = None,
    use_camconf_camera: bool = False,
    camconf_dir: str = None,
    camconf_angle_order=(0, 90, 180, 270),
    strict_camconf: bool = True,
    invert_camera_direction: bool = False,
) -> dict:
    target_dir = Path(target_dir)
    image_names = sorted(glob.glob(str(target_dir / 'images' / '*')))
    if len(image_names) == 0:
        raise ValueError(f'No images found in {target_dir / "images"}')

    images = load_and_preprocess_images(image_names).to(device)

    with torch.no_grad():
        if torch.cuda.is_available():
            with torch.cuda.amp.autocast(dtype=amp_dtype):
                predictions = model(images)
        else:
            predictions = model(images)

    extrinsic, intrinsic = pose_encoding_to_extri_intri(predictions['pose_enc'], images.shape[-2:])
    predictions['extrinsic'] = extrinsic
    predictions['intrinsic'] = intrinsic

    for key in list(predictions.keys()):
        if isinstance(predictions[key], torch.Tensor):
            predictions[key] = predictions[key].detach().cpu().numpy().squeeze(0)

    predictions['pose_enc_list'] = None
    predictions['camera_source'] = 'prediction'

    if use_camconf_camera:
        if image_path is None:
            raise ValueError('use_camconf_camera=True 时必须提供 image_path')
        if camconf_dir is None:
            raise ValueError('use_camconf_camera=True 时必须提供 camconf_dir')

        try:
            H, W = int(predictions['depth'].shape[1]), int(predictions['depth'].shape[2])
            n_pred = int(predictions['depth'].shape[0])

            pred_extr_before = predictions['extrinsic'].copy()
            world_points_before = unproject_depth_map_to_point_map(
                predictions['depth'],
                pred_extr_before,
                predictions['intrinsic']
            )
            if globals().get('debug_print_camera', False):
                point_center_before, forward_vecs_before, to_center_vecs_before, angles_deg_before = _camera_forward_angle_to_point_center(
                    pred_extr_before, world_points_before
                )
                print('[camera-debug] 变换前：相机前向(-Z)与点云中心夹角(前4):')
                print(f'  point_center_before=({point_center_before[0]:.4f}, {point_center_before[1]:.4f}, {point_center_before[2]:.4f})')
                for i in range(min(4, len(forward_vecs_before))):
                    fwd = forward_vecs_before[i]
                    vec = to_center_vecs_before[i]
                    ang = angles_deg_before[i]
                    print(
                        f'  cam_before[{i}] forward=({fwd[0]:.4f}, {fwd[1]:.4f}, {fwd[2]:.4f}) '
                        f'to_center=({vec[0]:.4f}, {vec[1]:.4f}, {vec[2]:.4f}) '
                        f'angle_deg={ang:.2f}'
                    )

            # 只用 0 度相机作为锚点；其他相机由预测相机整体刚体变换得到
            anchor_angle = int(camconf_angle_order[0]) if len(camconf_angle_order) > 0 else 0
            anc_ext44, anc_intr33, anc_json = load_camconf_anchor_camera_for_image(
                image_path=image_path,
                camconf_dir=camconf_dir,
                anchor_angle=anchor_angle,
                H=H,
                W=W,
            )

            ext_cam = _rigid_align_all_cameras_by_cam0(predictions['extrinsic'], anc_ext44)

            # 注意：这里不再应用“逐相机朝向翻转”，否则会改变几何
            if invert_camera_direction:
                print('[Warn] invert_camera_direction=True 会改变几何，已自动忽略。')

            # 只用锚点相机内参，复制到所有视角
            intr_cam = np.repeat(anc_intr33[None, ...], n_pred, axis=0).astype(np.float32)

            predictions['extrinsic'] = ext_cam.astype(np.float32)
            predictions['intrinsic'] = intr_cam
            predictions['camera_source'] = 'camconf_anchor0_rigid'
            predictions['camera_json_paths'] = np.array([anc_json], dtype=object)

            if globals().get('debug_print_camera', False):
                print(f'\n[camera-debug] image={image_path.name}')
                pred_centers_before = _camera_centers_from_extr(pred_extr_before)
                pred_centers_after = _camera_centers_from_extr(predictions['extrinsic'])
                pred_eulers_before = _camera_euler_xyz_deg_from_extr(pred_extr_before)
                pred_eulers_after = _camera_euler_xyz_deg_from_extr(predictions['extrinsic'])

                print('[camera-debug] 变换前预测相机中心+角度(前4):')
                for i in range(min(4, len(pred_centers_before))):
                    c = pred_centers_before[i]
                    e = pred_eulers_before[i]
                    print(
                        f'  pred_before[{i}] center=({c[0]:.4f}, {c[1]:.4f}, {c[2]:.4f}) '
                        f'euler_xyz_deg=({e[0]:.2f}, {e[1]:.2f}, {e[2]:.2f})'
                    )

                print('[camera-debug] anchor-rigid后相机中心+角度(前4，尚未post-align):')
                for i in range(min(4, len(pred_centers_after))):
                    c = pred_centers_after[i]
                    e = pred_eulers_after[i]
                    print(
                        f'  pred_after [{i}] center=({c[0]:.4f}, {c[1]:.4f}, {c[2]:.4f}) '
                        f'euler_xyz_deg=({e[0]:.2f}, {e[1]:.2f}, {e[2]:.2f})'
                    )

                # 文件中：打印锚点 + 若存在则打印同组 4 个角度
                base_stem = extract_camconf_base_stem(image_path.stem)
                print('[camera-debug] 文件中相机中心+角度:')
                for a in camconf_angle_order[:4]:
                    p = Path(camconf_dir) / f'{base_stem}_{a}_camconf.json'
                    if p.exists():
                        ext44, _ = parse_json_camera_extrinsic_intrinsic(
                            str(p), device='cpu', H=H, W=W, dtype=torch.float32, flip_y=True
                        )
                        ext_np = ext44.squeeze(0).detach().cpu().numpy()[None, ...]
                        cc = _camera_centers_from_extr(ext_np)[0]
                        ee = _camera_euler_xyz_deg_from_extr(ext_np)[0]
                        print(
                            f'  file[{a:>3}°] center=({cc[0]:.4f}, {cc[1]:.4f}, {cc[2]:.4f}) '
                            f'euler_xyz_deg=({ee[0]:.2f}, {ee[1]:.2f}, {ee[2]:.2f})'
                        )
                    else:
                        print(f'  file[{a:>3}°] = <missing>')
                print('')

            print(
                f'[camera] anchor={Path(anc_json).name} | mode=anchor0+rigid | extrinsic shape={predictions["extrinsic"].shape} | invert_applied=False'
            )

        except Exception as e:
            if strict_camconf:
                raise
            print(f'[Warn] camconf failed, fallback to prediction for {image_path.name}: {e}')

    depth_map = predictions['depth']
    world_points = unproject_depth_map_to_point_map(
        depth_map,
        predictions['extrinsic'],
        predictions['intrinsic']
    )
    predictions['world_points_from_depth'] = world_points
    if globals().get('debug_print_camera', False):
        point_center, forward_vecs, to_center_vecs, angles_deg = _camera_forward_angle_to_point_center(
            predictions['extrinsic'], world_points
        )
        print('[camera-debug] 相机前向(-Z)与点云中心夹角(前4):')
        print(f'  point_center=({point_center[0]:.4f}, {point_center[1]:.4f}, {point_center[2]:.4f})')
        for i in range(min(4, len(forward_vecs))):
            fwd = forward_vecs[i]
            vec = to_center_vecs[i]
            ang = angles_deg[i]
            print(
                f'  cam[{i}] forward=({fwd[0]:.4f}, {fwd[1]:.4f}, {fwd[2]:.4f}) '
                f'to_center=({vec[0]:.4f}, {vec[1]:.4f}, {vec[2]:.4f}) '
                f'angle_deg={ang:.2f}'
            )
    pts = np.asarray(predictions['world_points_from_depth'], dtype=np.float32).reshape(-1, 3)
    print(f'[DIAG] pts shape={pts.shape}')
    print(f'[DIAG] pts min={pts.min(axis=0)}, max={pts.max(axis=0)}, mean={pts.mean(axis=0)}')
    print(f'[DIAG] pts span (p95-p5)={_robust_span_diag(pts)}')
    print(f'[DIAG] cameras centers: {_camera_centers_from_extr(predictions["extrinsic"])}')
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return predictions

## 6) 将预测结果按图片名导出到 `output_path`（`.glb` / 可选 `.npz`）

In [7]:
def _normalize_depth_for_save(depth: np.ndarray) -> np.ndarray:
    d = np.asarray(depth, dtype=np.float32)

    # 支持形状：
    # (H, W) -> (1, H, W)
    # (N, H, W) -> (N, H, W)
    # (N, H, W, 1) -> (N, H, W)
    # (1, N, H, W) -> (N, H, W)（兜底）
    if d.ndim == 2:
        d = d[None, ...]
    elif d.ndim == 3:
        pass
    elif d.ndim == 4 and d.shape[-1] == 1:
        d = d[..., 0]
    elif d.ndim == 4 and d.shape[0] == 1:
        d = d[0]
    else:
        raise ValueError(f'Unexpected depth shape: {d.shape}')

    if d.ndim != 3:
        raise ValueError(f'Unexpected normalized depth shape: {d.shape}')

    return d.astype(np.float32, copy=False)


def save_raw_depth(
    predictions: dict,
    depth_output_dir: str,
    output_stem: str,
):
    depth_dir = Path(depth_output_dir)
    depth_dir.mkdir(parents=True, exist_ok=True)

    depth = _normalize_depth_for_save(predictions['depth'])

    raw_npy_path = depth_dir / f'{output_stem}_depth.npy'
    np.save(raw_npy_path, depth)

    return {
        'depth_npy': str(raw_npy_path),
        'depth_shape': tuple(depth.shape),
    }


def export_npz_and_glb(
    sample_target_dir: str,
    predictions: dict,
    output_dir: str,
    output_stem: str,
    conf_thres: float = 50.0,
    prediction_mode: str = 'Depthmap and Camera Branch',
    save_npz: bool = True,
    save_raw_depth_npy: bool = True,
    depth_output_dir: str = None,
    only_output_depth: bool = False,
):
    sample_target_dir = Path(sample_target_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    glb_path = output_dir / f'{output_stem}.glb'
    npz_path = output_dir / f'{output_stem}.npz'

    depth_outputs = None
    if save_raw_depth_npy:
        if depth_output_dir is None:
            depth_output_dir = f"{str(sample_target_dir).rstrip('/')}_depth"
        depth_outputs = save_raw_depth(
            predictions=predictions,
            depth_output_dir=depth_output_dir,
            output_stem=output_stem,
        )

    if only_output_depth:
        return None, None, depth_outputs

    if save_npz:
        np.savez(npz_path, **predictions)

    glbscene = predictions_to_glb(
        predictions,
        conf_thres=conf_thres,
        filter_by_frames='All',
        show_cam=True,
        mask_sky=False,
        mask_black_bg=False,
        mask_white_bg=False,
        target_dir=str(sample_target_dir),
        prediction_mode=prediction_mode,
    )
    glbscene.export(file_obj=str(glb_path))

    return (str(npz_path) if save_npz else None), str(glb_path), depth_outputs

In [8]:
def _to_extr44(extrinsic: np.ndarray) -> np.ndarray:
    ext = np.asarray(extrinsic, dtype=np.float32)
    if ext.ndim != 3:
        raise ValueError(f'extrinsic ndim error: {ext.shape}')
    if ext.shape[1:] == (4, 4):
        return ext
    if ext.shape[1:] == (3, 4):
        out = np.tile(np.eye(4, dtype=np.float32), (ext.shape[0], 1, 1))
        out[:, :3, :4] = ext
        return out
    raise ValueError(f'unsupported extrinsic shape: {ext.shape}')


def _from_extr44(ext44: np.ndarray, like_extr: np.ndarray) -> np.ndarray:
    like = np.asarray(like_extr)
    if like.shape[1:] == (3, 4):
        return ext44[:, :3, :4].astype(np.float32)
    if like.shape[1:] == (4, 4):
        return ext44.astype(np.float32)
    raise ValueError(f'unsupported like_extr shape: {like.shape}')


def _rotation_align_vec_to_vec(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a = a / (np.linalg.norm(a) + 1e-8)
    b = b / (np.linalg.norm(b) + 1e-8)
    v = np.cross(a, b)
    c = float(np.dot(a, b))
    s = float(np.linalg.norm(v))
    if s < 1e-8:
        if c > 0:
            return np.eye(3, dtype=np.float32)
        axis = np.array([1.0, 0.0, 0.0], dtype=np.float32)
        if abs(a[0]) > 0.9:
            axis = np.array([0.0, 1.0, 0.0], dtype=np.float32)
        v = np.cross(a, axis)
        v = v / (np.linalg.norm(v) + 1e-8)
        K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]], dtype=np.float32)
        return (np.eye(3, dtype=np.float32) + 2 * (K @ K)).astype(np.float32)

    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]], dtype=np.float32)
    R = np.eye(3, dtype=np.float32) + K + K @ K * ((1 - c) / (s * s + 1e-8))
    return R.astype(np.float32)


def _estimate_similarity_umeyama(src: np.ndarray, dst: np.ndarray, with_scale: bool = True):
    """
    求解 s, R, t，使 dst ≈ s * R * src + t
    src/dst: (N,3)
    """
    src = np.asarray(src, dtype=np.float32)
    dst = np.asarray(dst, dtype=np.float32)
    if src.shape != dst.shape or src.ndim != 2 or src.shape[1] != 3:
        raise ValueError(f'bad shapes: src={src.shape}, dst={dst.shape}')

    mu_s = src.mean(axis=0)
    mu_d = dst.mean(axis=0)
    X = src - mu_s[None, :]
    Y = dst - mu_d[None, :]

    cov = (Y.T @ X) / src.shape[0]
    U, D, Vt = np.linalg.svd(cov)
    S = np.eye(3, dtype=np.float32)
    if np.linalg.det(U @ Vt) < 0:
        S[-1, -1] = -1.0
    R = (U @ S @ Vt).astype(np.float32)

    if with_scale:
        var_s = np.mean(np.sum(X * X, axis=1)) + 1e-8
        scale = float(np.sum(D * np.diag(S)) / var_s)
    else:
        scale = 1.0

    t = (mu_d - scale * (R @ mu_s)).astype(np.float32)
    return scale, R, t


def _apply_similarity_to_predictions(predictions: dict, s: float, R: np.ndarray, t: np.ndarray) -> dict:
    # points
    wp = np.asarray(predictions['world_points_from_depth'], dtype=np.float32)
    pts = wp.reshape(-1, 3)
    pts_new = (s * (pts @ R.T) + t[None, :]).astype(np.float32)
    predictions['world_points_from_depth'] = pts_new.reshape(wp.shape)

    # cameras
    ext_old = np.asarray(predictions['extrinsic'], dtype=np.float32)
    ext44 = _to_extr44(ext_old)
    c2w = np.linalg.inv(ext44)
    c2w_new = c2w.copy()
    c2w_new[:, :3, :3] = np.einsum('ij,njk->nik', R, c2w[:, :3, :3])
    c2w_new[:, :3, 3] = s * np.einsum('ij,nj->ni', R, c2w[:, :3, 3]) + t[None, :]
    ext_new44 = np.linalg.inv(c2w_new).astype(np.float32)
    predictions['extrinsic'] = _from_extr44(ext_new44, ext_old)
    return predictions


def _apply_similarity_to_points_only(predictions: dict, s: float, R: np.ndarray, t: np.ndarray) -> dict:
    wp = np.asarray(predictions['world_points_from_depth'], dtype=np.float32)
    pts = wp.reshape(-1, 3)
    pts_new = (s * (pts @ R.T) + t[None, :]).astype(np.float32)
    predictions['world_points_from_depth'] = pts_new.reshape(wp.shape)
    return predictions


def _robust_span_diag(points: np.ndarray, q_low: float = 5.0, q_high: float = 95.0) -> float:
    p = np.asarray(points, dtype=np.float32)
    if p.ndim != 2 or p.shape[1] != 3 or p.shape[0] == 0:
        return 0.0
    lo = np.percentile(p, q_low, axis=0)
    hi = np.percentile(p, q_high, axis=0)
    return float(np.linalg.norm(hi - lo))


def _load_file_camera_centers_for_image(
    image_path: Path,
    camconf_dir: str,
    angle_order=(0, 90, 180, 270),
    H: int = 256,
    W: int = 256,
):
    base_stem = extract_camconf_base_stem(image_path.stem)
    centers = []
    valid_angles = []
    for a in angle_order:
        p = Path(camconf_dir) / f'{base_stem}_{int(a)}_camconf.json'
        if not p.exists():
            continue
        ext44, _ = parse_json_camera_extrinsic_intrinsic(
            str(p), device='cpu', H=H, W=W, dtype=torch.float32, flip_y=True
        )
        c = _camera_centers_from_extr(ext44.squeeze(0).detach().cpu().numpy()[None, ...])[0]
        centers.append(c.astype(np.float32))
        valid_angles.append(int(a))

    if len(centers) == 0:
        raise FileNotFoundError(f'no camconf found for {image_path.name} in {camconf_dir}')
    return np.stack(centers, axis=0), valid_angles


def apply_global_rigid_similarity(
    predictions: dict,
    mode: str = 'rigid',
    scale: float = 1.0,
    image_path: Path = None,
    camconf_dir: str = None,
    camconf_angle_order=(0, 90, 180, 270),
    camera_fit_match_range: bool = True,
    camera_fit_range_gain: float = 1.0,
    camera_fit_scale_min: float = 0.25,
    camera_fit_scale_max: float = 4.0,
    camera_fit_range_apply_to_camera: bool = True,
 ) -> dict:
    """
    对点云和相机施加全局变换。
    mode: none | rigid | similarity | camera_fit
    """
    mode = (mode or 'none').lower()
    if mode == 'none':
        return predictions

    if 'world_points_from_depth' not in predictions or 'extrinsic' not in predictions:
        return predictions

    if mode in ('rigid', 'similarity'):
        wp = np.asarray(predictions['world_points_from_depth'], dtype=np.float32)
        pts = wp.reshape(-1, 3)

        if 'depth_conf' in predictions:
            conf = np.asarray(predictions['depth_conf'], dtype=np.float32).reshape(-1)
            th = np.percentile(conf, 60.0)
            m = conf >= th
            fit_pts = pts[m] if np.any(m) else pts
        else:
            fit_pts = pts

        center = fit_pts.mean(axis=0)
        X = fit_pts - center[None, :]
        _, _, vh = np.linalg.svd(X, full_matrices=False)
        normal = vh[-1]
        if normal[2] < 0:
            normal = -normal

        Rg = _rotation_align_vec_to_vec(normal.astype(np.float32), np.array([0.0, 0.0, 1.0], dtype=np.float32))
        s = 1.0 if mode == 'rigid' else float(scale)
        if s <= 0:
            s = 1.0

        t = (-s * (Rg @ center)).astype(np.float32)
        predictions = _apply_similarity_to_predictions(predictions, s=s, R=Rg, t=t)
        print(f'[post-align] mode={mode}, scale={s:.4f} (global transform applied to points+cameras)')
        return predictions

    if mode == 'camera_fit':
        if image_path is None or camconf_dir is None:
            print('[post-align] camera_fit missing image_path/camconf_dir, fallback to rigid')
            return apply_global_rigid_similarity(predictions, mode='rigid', scale=1.0)

        H, W = int(predictions['depth'].shape[1]), int(predictions['depth'].shape[2])
        file_centers, valid_angles = _load_file_camera_centers_for_image(
            image_path=image_path,
            camconf_dir=camconf_dir,
            angle_order=camconf_angle_order,
            H=H,
            W=W,
        )

        pred_centers = _camera_centers_from_extr(np.asarray(predictions['extrinsic'], dtype=np.float32))
        n = min(len(pred_centers), len(file_centers))
        src = pred_centers[:n].astype(np.float32)
        dst = file_centers[:n].astype(np.float32)

        s_cam, R_cam, t_cam = _estimate_similarity_umeyama(src, dst, with_scale=True)
        predictions = _apply_similarity_to_predictions(predictions, s=s_cam, R=R_cam, t=t_cam)

        wp2 = np.asarray(predictions['world_points_from_depth'], dtype=np.float32)
        pts2 = wp2.reshape(-1, 3)
        pts_center = pts2.mean(axis=0)
        tgt_xy = dst[:, :2].mean(axis=0)
        target_center = np.array([tgt_xy[0], tgt_xy[1], 0.0], dtype=np.float32)
        delta = (target_center - pts_center).astype(np.float32)
        # 同步平移：点云和相机同时平移，避免朝向判定因相对位姿改变而翻转
        predictions = _apply_similarity_to_predictions(
            predictions, s=1.0, R=np.eye(3, dtype=np.float32), t=delta
        )

        range_scale_applied = 1.0
        file_span = _robust_span_diag(dst, q_low=5.0, q_high=95.0)
        pred_span_before = _robust_span_diag(
            np.asarray(predictions['world_points_from_depth'], dtype=np.float32).reshape(-1, 3),
            q_low=5.0,
            q_high=95.0,
        )
        if camera_fit_match_range and file_span > 1e-6 and pred_span_before > 1e-6:
            raw_s = file_span / pred_span_before
            gain = float(camera_fit_range_gain)
            mixed_s = 1.0 + gain * (raw_s - 1.0)
            mixed_s = float(np.clip(mixed_s, camera_fit_scale_min, camera_fit_scale_max))

            t_scale = ((1.0 - mixed_s) * target_center).astype(np.float32)
            if camera_fit_range_apply_to_camera:
                predictions = _apply_similarity_to_predictions(
                    predictions, s=mixed_s, R=np.eye(3, dtype=np.float32), t=t_scale
                )
            else:
                predictions = _apply_similarity_to_points_only(
                    predictions, s=mixed_s, R=np.eye(3, dtype=np.float32), t=t_scale
                )
            range_scale_applied = mixed_s

        pred_span_after = _robust_span_diag(
            np.asarray(predictions['world_points_from_depth'], dtype=np.float32).reshape(-1, 3),
            q_low=5.0,
            q_high=95.0,
        )

        pred_after = _camera_centers_from_extr(np.asarray(predictions['extrinsic'], dtype=np.float32))[:n]
        rms_before = float(np.sqrt(np.mean(np.sum((src - dst) ** 2, axis=1))))
        rms_after = float(np.sqrt(np.mean(np.sum((pred_after - dst) ** 2, axis=1))))
        print(
            f'[post-align] mode=camera_fit, used_angles={valid_angles[:n]}, '
            f'cam_scale={s_cam:.4f}, range_scale={range_scale_applied:.4f}, '
            f'range_apply_to_camera={camera_fit_range_apply_to_camera}, '
            f'rms_before={rms_before:.4f}, rms_after={rms_after:.4f}'
        )
        print(
            f'[post-align] span(file_cam)={file_span:.4f}, '
            f'span(pred_pts_before)={pred_span_before:.4f}, span(pred_pts_after)={pred_span_after:.4f}'
        )
        if globals().get('debug_print_camera', False):
            print('[camera-debug] post-align后相机中心(前4):')
            for i in range(min(4, n)):
                c = pred_after[i]
                f = dst[i]
                print(
                    f'  post[{i}] center=({c[0]:.4f}, {c[1]:.4f}, {c[2]:.4f}) | '
                    f'file=({f[0]:.4f}, {f[1]:.4f}, {f[2]:.4f})'
                )
        print(f'[post-align] point_center -> ({target_center[0]:.4f}, {target_center[1]:.4f}, 0.0000)')
        return predictions

    print(f'[post-align] unknown mode={mode}, skip')
    return predictions

## 7) 批量执行：支持 `input_path` 为单图或文件夹，结果保存到 `output_path`

In [9]:
def reconstruct_one_image(
    image_path: Path,
    output_dir: Path,
    conf_thres: float,
    prediction_mode: str,
    force_resize: bool,
    save_npz: bool,
    save_raw_depth_npy: bool = True,
    depth_output_dir: str = None,
    only_output_depth: bool = False,
    use_camconf_camera: bool = False,
    camconf_dir: str = None,
    camconf_angle_order=(0, 90, 180, 270),
    strict_camconf: bool = True,
    invert_camera_direction: bool = False,
    post_align_mode: str = 'none',
    post_align_scale: float = 1.0,
    camera_fit_match_range: bool = True,
    camera_fit_range_gain: float = 1.0,
    camera_fit_scale_min: float = 0.25,
    camera_fit_scale_max: float = 4.0,
    camera_fit_range_apply_to_camera: bool = True,
):
    img_rgb = read_single_image_512(str(image_path), force_resize=force_resize)

    # 每张图独立临时目录，避免相互覆盖
    sample_target_dir = Path(tempfile.mkdtemp(prefix=f'vggt_{image_path.stem}_'))
    _ = split_512_to_4x256_and_save(img_rgb, sample_target_dir)

    predictions = run_model(
        target_dir=sample_target_dir,
        model=model,
        image_path=image_path,
        use_camconf_camera=use_camconf_camera,
        camconf_dir=camconf_dir,
        camconf_angle_order=camconf_angle_order,
        strict_camconf=strict_camconf,
        invert_camera_direction=invert_camera_direction,
    )

    predictions = apply_global_rigid_similarity(
        predictions,
        mode=post_align_mode,
        scale=post_align_scale,
        image_path=image_path,
        camconf_dir=camconf_dir,
        camconf_angle_order=camconf_angle_order,
        camera_fit_match_range=camera_fit_match_range,
        camera_fit_range_gain=camera_fit_range_gain,
        camera_fit_scale_min=camera_fit_scale_min,
        camera_fit_scale_max=camera_fit_scale_max,
        camera_fit_range_apply_to_camera=camera_fit_range_apply_to_camera,
    )

    npz_path, glb_path, depth_outputs = export_npz_and_glb(
        sample_target_dir=sample_target_dir,
        predictions=predictions,
        output_dir=output_dir,
        output_stem=image_path.stem,
        conf_thres=conf_thres,
        prediction_mode=prediction_mode,
        save_npz=save_npz,
        save_raw_depth_npy=save_raw_depth_npy,
        depth_output_dir=depth_output_dir,
        only_output_depth=only_output_depth,
    )

    return {
        'input_image': str(image_path),
        'target_dir': str(sample_target_dir),
        'glb_path': glb_path,
        'npz_path': npz_path,
        'depth_outputs': depth_outputs,
        'camera_source': predictions.get('camera_source', 'prediction'),
    }


def reconstruct_from_input_path(
    input_path: str,
    output_path: str,
    conf_thres: float = 50.0,
    prediction_mode: str = 'Depthmap and Camera Branch',
    force_resize: bool = True,
    save_npz: bool = True,
    save_raw_depth_npy: bool = True,
    depth_output_dir: str = None,
    only_output_depth: bool = False,
    valid_exts=None,
    use_camconf_camera: bool = False,
    camconf_dir: str = None,
    camconf_angle_order=(0, 90, 180, 270),
    strict_camconf: bool = True,
    invert_camera_direction: bool = False,
    post_align_mode: str = 'none',
    post_align_scale: float = 1.0,
    camera_fit_match_range: bool = True,
    camera_fit_range_gain: float = 1.0,
    camera_fit_scale_min: float = 0.25,
    camera_fit_scale_max: float = 4.0,
    camera_fit_range_apply_to_camera: bool = True,
):
    t0 = time.time()
    output_dir = Path(output_path)
    output_dir.mkdir(parents=True, exist_ok=True)

    if depth_output_dir is None:
        depth_output_dir = f"{str(input_path).rstrip('/')}_depth"
    Path(depth_output_dir).mkdir(parents=True, exist_ok=True)
    print('depth_output_dir =', depth_output_dir)

    images = collect_input_images(input_path, valid_exts=valid_exts)
    print(f'[Start] total images: {len(images)}')

    results = []
    for i, img_path in enumerate(images, 1):
        print(f'[{i}/{len(images)}] processing: {img_path.name}')
        item = reconstruct_one_image(
            image_path=img_path,
            output_dir=output_dir,
            conf_thres=conf_thres,
            prediction_mode=prediction_mode,
            force_resize=force_resize,
            save_npz=save_npz,
            save_raw_depth_npy=save_raw_depth_npy,
            depth_output_dir=depth_output_dir,
            only_output_depth=only_output_depth,
            use_camconf_camera=use_camconf_camera,
            camconf_dir=camconf_dir,
            camconf_angle_order=camconf_angle_order,
            strict_camconf=strict_camconf,
            invert_camera_direction=invert_camera_direction,
            post_align_mode=post_align_mode,
            post_align_scale=post_align_scale,
            camera_fit_match_range=camera_fit_match_range,
            camera_fit_range_gain=camera_fit_range_gain,
            camera_fit_scale_min=camera_fit_scale_min,
            camera_fit_scale_max=camera_fit_scale_max,
            camera_fit_range_apply_to_camera=camera_fit_range_apply_to_camera,
        )
        results.append(item)

    elapsed = time.time() - t0
    print(f'[Done] {len(results)} image(s), elapsed: {elapsed:.2f}s')
    print('output_dir =', output_dir)
    return results

## 8) 一键运行（只保存，不显示 GLB）

In [10]:
import builtins
import numpy as np
import re

# 注意：请先运行“第18单元”和“第16单元”恢复原始函数，再运行本单元
_base_reconstruct_one_image = reconstruct_one_image
_base_apply_global_rigid_similarity = apply_global_rigid_similarity


def _facing_label(angle_deg: float) -> str:
    if angle_deg < 30.0:
        return '朝向点云'
    if angle_deg < 90.0:
        return '斜向点云'
    if angle_deg < 150.0:
        return '偏离点云'
    return '背向点云'


def apply_global_rigid_similarity(*args, **kwargs):
    """保留三组打印里的最后一组，并补充“相机是否朝向点云”的解释性输出。"""
    orig_print = builtins.print

    def filtered_print(*p_args, **p_kwargs):
        text = ' '.join(str(x) for x in p_args)
        if '[camera-debug] post-align后相机中心(前4):' in text:
            return
        if 'post[' in text:
            return
        if '[post-align] point_center ->' in text:
            return
        return orig_print(*p_args, **p_kwargs)

    builtins.print = filtered_print
    try:
        predictions = _base_apply_global_rigid_similarity(*args, **kwargs)
    finally:
        builtins.print = orig_print

    mode = (kwargs.get('mode', 'rigid') or 'none').lower()
    image_path = kwargs.get('image_path', None)
    camconf_dir = kwargs.get('camconf_dir', None)
    camconf_angle_order = kwargs.get('camconf_angle_order', (0, 90, 180, 270))

    if mode == 'camera_fit' and image_path is not None and camconf_dir is not None and globals().get('debug_print_camera', False):
        H = int(predictions['depth'].shape[1])
        W = int(predictions['depth'].shape[2])
        file_centers, _ = _load_file_camera_centers_for_image(
            image_path=image_path,
            camconf_dir=camconf_dir,
            angle_order=camconf_angle_order,
            H=H,
            W=W,
        )

        pred_after = _camera_centers_from_extr(np.asarray(predictions['extrinsic'], dtype=np.float32))
        pred_after_eulers = _camera_euler_xyz_deg_from_extr(np.asarray(predictions['extrinsic'], dtype=np.float32))
        n = min(len(pred_after), len(file_centers))

        print('[camera-debug] post-align最终相机中心+角度(前4):')
        for i in range(min(4, n)):
            c = pred_after[i]
            e = pred_after_eulers[i]
            f = file_centers[i]
            print(
                f'  post[{i}] center=({c[0]:.4f}, {c[1]:.4f}, {c[2]:.4f}) '
                f'euler_xyz_deg=({e[0]:.2f}, {e[1]:.2f}, {e[2]:.2f}) | '
                f'file=({f[0]:.4f}, {f[1]:.4f}, {f[2]:.4f})'
            )

        # 额外：解释“是否朝向点云”
        point_center, _, _, angles_deg = _camera_forward_angle_to_point_center(
            np.asarray(predictions['extrinsic'], dtype=np.float32),
            np.asarray(predictions['world_points_from_depth'], dtype=np.float32),
        )
        k = min(4, len(angles_deg))
        if k > 0:
            sel = angles_deg[:k]
            ok = int(np.sum(sel < 60.0))
            print('[camera-help] 角度定义：相机前向(-Z) 与 (相机中心->点云中心) 的夹角。')
            print('[camera-help] 判定：<30°朝向点云；30~90°斜向点云；>90°偏离；接近180°背向点云。')
            print(
                f'[camera-help] post-align前{k}个相机: mean={float(np.mean(sel)):.2f}°, '
                f'min={float(np.min(sel)):.2f}°, max={float(np.max(sel)):.2f}°, '
                f'ok(<60°)={ok}/{k}, point_center=({point_center[0]:.4f}, {point_center[1]:.4f}, {point_center[2]:.4f})'
            )
            for i in range(k):
                a = float(sel[i])
                print(f'  [camera-help] cam[{i}] angle={a:.2f}° -> {_facing_label(a)}')

    return predictions


def reconstruct_one_image(*args, **kwargs):
    """只保留三组相机打印：变换前 / 文件参考 / 最终 post-align。"""
    orig_print = builtins.print

    before_angles = []
    before_point_center = None

    def filtered_print(*p_args, **p_kwargs):
        nonlocal before_point_center
        text = ' '.join(str(x) for x in p_args)

        if '[camera-help]' in text:
            return orig_print(*p_args, **p_kwargs)

        # 采集“变换前是否朝向点云”所需信息（仍然隐藏原始冗长行）
        if 'point_center_before=' in text:
            m = re.search(r'point_center_before=\(([^\)]+)\)', text)
            if m:
                vals = [v.strip() for v in m.group(1).split(',')]
                if len(vals) == 3:
                    try:
                        before_point_center = (float(vals[0]), float(vals[1]), float(vals[2]))
                    except Exception:
                        before_point_center = None
            return

        if 'cam_before[' in text and 'angle_deg=' in text:
            m = re.search(r'angle_deg=([-+]?\d*\.?\d+)', text)
            if m:
                try:
                    before_angles.append(float(m.group(1)))
                except Exception:
                    pass
            return

        if '[camera-debug] 变换前：相机前向(-Z)与点云中心夹角(前4):' in text:
            return
        if '[camera-debug] 相机前向(-Z)与点云中心夹角(前4):' in text:
            return
        if 'point_center=' in text and 'point_center_before=' not in text:
            return
        if 'cam[' in text and 'cam_before[' not in text:
            return
        if '[camera-debug] anchor-rigid后相机中心+角度(前4，尚未post-align):' in text:
            return
        if 'pred_after [' in text:
            return
        return orig_print(*p_args, **p_kwargs)

    builtins.print = filtered_print
    try:
        result = _base_reconstruct_one_image(*args, **kwargs)
    finally:
        builtins.print = orig_print

    if globals().get('debug_print_camera', False) and len(before_angles) > 0:
        k = min(4, len(before_angles))
        sel = np.asarray(before_angles[:k], dtype=np.float32)
        ok = int(np.sum(sel < 60.0))
        if before_point_center is None:
            pc_str = 'N/A'
        else:
            pc_str = f'({before_point_center[0]:.4f}, {before_point_center[1]:.4f}, {before_point_center[2]:.4f})'

        print('[camera-help] 角度定义：相机前向(-Z) 与 (相机中心->点云中心) 的夹角。')
        print('[camera-help] 判定：<30°朝向点云；30~90°斜向点云；>90°偏离；接近180°背向点云。')
        print(
            f'[camera-help] 变换前前{k}个相机: mean={float(np.mean(sel)):.2f}°, '
            f'min={float(np.min(sel)):.2f}°, max={float(np.max(sel)):.2f}°, '
            f'ok(<60°)={ok}/{k}, point_center_before={pc_str}'
        )
        for i in range(k):
            a = float(sel[i])
            print(f'  [camera-help] before cam[{i}] angle={a:.2f}° -> {_facing_label(a)}')

    return result


In [11]:
# === 精确相机约束模式（推荐）===
# 目标：让内外参与 camconf 尽可能一致，同时只调整点云，不再改相机

EXACT_CAMCONF_MODE = True
EXACT_POINT_SCALE_TO_CAMCONF = True

_required = ['run_model', 'apply_global_rigid_similarity', '_apply_similarity_to_points_only']
_missing = [k for k in _required if k not in globals()]
if _missing:
    print('[EXACT_CAMCONF_MODE] 请先运行前面的函数定义单元，再运行本单元。缺少:', _missing)
else:
    _base_run_model_exact = run_model
    _base_apply_global_exact = apply_global_rigid_similarity

    def _load_camconf_all_cameras_for_image(
        image_path: Path,
        camconf_dir: str,
        angle_order=(0, 90, 180, 270),
        H: int = 256,
        W: int = 256,
    ):
        base_stem = extract_camconf_base_stem(image_path.stem)
        exts, intrs, paths, valid_angles = [], [], [], []
        for a in angle_order:
            p = Path(camconf_dir) / f'{base_stem}_{int(a)}_camconf.json'
            if not p.exists():
                continue
            ext44, intr33 = parse_json_camera_extrinsic_intrinsic(
                str(p), device='cpu', H=H, W=W, dtype=torch.float32, flip_y=True
            )
            exts.append(ext44.squeeze(0).detach().cpu().numpy().astype(np.float32))
            intrs.append(intr33.squeeze(0).detach().cpu().numpy().astype(np.float32))
            paths.append(str(p))
            valid_angles.append(int(a))

        if len(exts) == 0:
            raise FileNotFoundError(f'no camconf found for {image_path.name} in {camconf_dir}')
        return np.stack(exts, axis=0), np.stack(intrs, axis=0), paths, valid_angles

    def run_model(
        target_dir: str,
        model,
        image_path: Path = None,
        use_camconf_camera: bool = False,
        camconf_dir: str = None,
        camconf_angle_order=(0, 90, 180, 270),
        strict_camconf: bool = True,
        invert_camera_direction: bool = False,
    ) -> dict:
        predictions = _base_run_model_exact(
            target_dir=target_dir,
            model=model,
            image_path=image_path,
            use_camconf_camera=False,
            camconf_dir=camconf_dir,
            camconf_angle_order=camconf_angle_order,
            strict_camconf=strict_camconf,
            invert_camera_direction=invert_camera_direction,
        )

        if not use_camconf_camera:
            return predictions

        if image_path is None or camconf_dir is None:
            raise ValueError('EXACT_CAMCONF_MODE 下 use_camconf_camera=True 需要 image_path 与 camconf_dir')

        try:
            H, W = int(predictions['depth'].shape[1]), int(predictions['depth'].shape[2])
            n_pred = int(predictions['depth'].shape[0])
            file_ext, file_intr, file_jsons, valid_angles = _load_camconf_all_cameras_for_image(
                image_path=image_path,
                camconf_dir=camconf_dir,
                angle_order=camconf_angle_order,
                H=H,
                W=W,
            )

            if file_ext.shape[0] < n_pred:
                raise ValueError(
                    f'camconf views not enough: need={n_pred}, got={file_ext.shape[0]}, image={image_path.name}'
                )

            # 外参形状适配到预测形状（通常是 3x4，避免导出 GLB 时 shape 冲突）
            ref_extr = np.asarray(predictions['extrinsic'], dtype=np.float32)
            file_ext_like = _convert_extrinsic_shape_like(file_ext[:n_pred], ref_extr)

            predictions['extrinsic'] = file_ext_like.astype(np.float32)
            predictions['intrinsic'] = file_intr[:n_pred].astype(np.float32)
            predictions['camera_source'] = 'camconf_exact'
            predictions['camera_json_paths'] = np.array(file_jsons[:n_pred], dtype=object)

            predictions['world_points_from_depth'] = unproject_depth_map_to_point_map(
                predictions['depth'], predictions['extrinsic'], predictions['intrinsic']
            )

            pred_cent = _camera_centers_from_extr(np.asarray(predictions['extrinsic'], dtype=np.float32))
            ref_cent = _camera_centers_from_extr(np.asarray(file_ext_like, dtype=np.float32))
            rms = float(np.sqrt(np.mean(np.sum((pred_cent - ref_cent) ** 2, axis=1))))
            print(
                f'[camera-exact] {image_path.name} | used_angles={valid_angles[:n_pred]} | '
                f'camera_rms={rms:.6f} (should be ~0)'
            )

        except Exception as e:
            if strict_camconf:
                raise
            print(f'[Warn] exact camconf failed, fallback to prediction for {image_path.name}: {e}')

        return predictions

    def apply_global_rigid_similarity(
        predictions: dict,
        mode: str = 'rigid',
        scale: float = 1.0,
        image_path: Path = None,
        camconf_dir: str = None,
        camconf_angle_order=(0, 90, 180, 270),
        camera_fit_match_range: bool = True,
        camera_fit_range_gain: float = 1.0,
        camera_fit_scale_min: float = 0.25,
        camera_fit_scale_max: float = 4.0,
        camera_fit_range_apply_to_camera: bool = True,
    ) -> dict:
        if predictions.get('camera_source', '') == 'camconf_exact':
            mode_l = (mode or 'none').lower()
            if mode_l == 'none':
                return predictions

            if not EXACT_POINT_SCALE_TO_CAMCONF:
                print('[post-align-exact] camera locked, skip post-align')
                return predictions

            if image_path is None or camconf_dir is None:
                print('[post-align-exact] missing image_path/camconf_dir, skip')
                return predictions

            H, W = int(predictions['depth'].shape[1]), int(predictions['depth'].shape[2])
            file_centers, valid_angles = _load_file_camera_centers_for_image(
                image_path=image_path,
                camconf_dir=camconf_dir,
                angle_order=camconf_angle_order,
                H=H,
                W=W,
            )

            wp = np.asarray(predictions['world_points_from_depth'], dtype=np.float32)
            pts = wp.reshape(-1, 3)
            pts_center = pts.mean(axis=0)
            tgt_xy = file_centers[:, :2].mean(axis=0)
            target_center = np.array([tgt_xy[0], tgt_xy[1], 0.0], dtype=np.float32)

            if mode_l == 'rigid':
                s = 1.0
            elif mode_l == 'similarity':
                s = float(scale) if float(scale) > 0 else 1.0
            else:
                pred_span = _robust_span_diag(pts, q_low=5.0, q_high=95.0)
                file_span = _robust_span_diag(file_centers, q_low=5.0, q_high=95.0)
                if camera_fit_match_range and pred_span > 1e-6 and file_span > 1e-6:
                    raw_s = file_span / pred_span
                    mixed_s = 1.0 + float(camera_fit_range_gain) * (raw_s - 1.0)
                    s = float(np.clip(mixed_s, camera_fit_scale_min, camera_fit_scale_max))
                else:
                    s = 1.0

            t = (target_center - s * pts_center).astype(np.float32)
            predictions = _apply_similarity_to_points_only(
                predictions, s=s, R=np.eye(3, dtype=np.float32), t=t
            )
            print(
                f'[post-align-exact] mode={mode_l}, used_angles={valid_angles}, '
                f'scale={s:.4f}, camera_locked=True (intr/extr unchanged)'
            )
            return predictions

        return _base_apply_global_exact(
            predictions=predictions,
            mode=mode,
            scale=scale,
            image_path=image_path,
            camconf_dir=camconf_dir,
            camconf_angle_order=camconf_angle_order,
            camera_fit_match_range=camera_fit_match_range,
            camera_fit_range_gain=camera_fit_range_gain,
            camera_fit_scale_min=camera_fit_scale_min,
            camera_fit_scale_max=camera_fit_scale_max,
            camera_fit_range_apply_to_camera=camera_fit_range_apply_to_camera,
        )

    print('[EXACT_CAMCONF_MODE] enabled: intr/extr will be locked to camconf when use_camconf_camera=True')

[EXACT_CAMCONF_MODE] enabled: intr/extr will be locked to camconf when use_camconf_camera=True


In [12]:
results = reconstruct_from_input_path(
    input_path=input_path,
    output_path=output_path,
    conf_thres=conf_thres,
    prediction_mode=prediction_mode,
    force_resize=force_resize,
    save_npz=save_npz,
    save_raw_depth_npy=save_raw_depth_npy,
    depth_output_dir=depth_output_path,
    only_output_depth=only_output_depth,
    valid_exts=valid_exts,
    use_camconf_camera=use_camconf_camera,
    camconf_dir=camconf_dir,
    camconf_angle_order=camconf_angle_order,
    strict_camconf=strict_camconf,
    invert_camera_direction=invert_camera_direction,
    post_align_mode=post_align_mode,
    post_align_scale=post_align_scale,
    camera_fit_match_range=camera_fit_match_range,
    camera_fit_range_gain=camera_fit_range_gain,
    camera_fit_scale_min=camera_fit_scale_min,
    camera_fit_scale_max=camera_fit_scale_max,
    camera_fit_range_apply_to_camera=camera_fit_range_apply_to_camera,
)

print('\n生成结果（前10条）:')
for row in results[:10]:
    depth_info = row.get('depth_outputs', None)
    if depth_info is None:
        depth_msg = 'depth=off'
    else:
        depth_msg = f"depth_npy={Path(depth_info.get('depth_npy', '')).name}"

    glb_name = Path(row['glb_path']).name if row.get('glb_path', None) else 'SKIPPED'
    print(
        f"- {Path(row['input_image']).name} -> {glb_name} "
        f"(camera={row.get('camera_source', 'prediction')}, {depth_msg})"
    )

# only_output_depth=True 时，glb_path/npz_path 为 None
results[:3]

depth_output_dir = /home/viplab/liukejia/project/result/render_3d_stitched_depth
[Start] total images: 9
[1/9] processing: num123_-800_100_0_0007.png
[DIAG] pts shape=(1073296, 3)
[DIAG] pts min=[-0.28479123 -0.26024848  0.8540226 ], max=[0.26843187 0.23174386 1.1541613 ], mean=[ 0.00103092 -0.03648996  0.9849603 ]
[DIAG] pts span (p95-p5)=0.5608378052711487
[DIAG] cameras centers: [[-9.3604402e-05 -4.6534795e-05  3.0325432e-04]
 [ 4.9613926e-01 -4.3133426e-01  2.5332913e-01]
 [-6.4753378e-03 -8.9428872e-01  5.1086628e-01]
 [-5.1288813e-01 -4.4012478e-01  2.4273330e-01]]
[camera-exact] num123_-800_100_0_0007.png | used_angles=[0, 90, 180, 270] | camera_rms=0.000000 (should be ~0)
[post-align-exact] mode=camera_fit, used_angles=[0, 90, 180, 270], scale=0.8455, camera_locked=True (intr/extr unchanged)
Building GLB scene
Using Depthmap and Camera Branch
GLB Scene built
[2/9] processing: num123_-800_150_0_0008.png
[DIAG] pts shape=(1073296, 3)
[DIAG] pts min=[-0.28195944 -0.28566995  0.847

[{'input_image': '/home/viplab/liukejia/project/result/render_3d_stitched/num123_-800_100_0_0007.png',
  'target_dir': '/tmp/vggt_num123_-800_100_0_0007_6gybf7bx',
  'glb_path': '/home/viplab/liukejia/project/result/reconstruct_pred_scale2/num123_-800_100_0_0007.glb',
  'npz_path': None,
  'depth_outputs': None,
  'camera_source': 'camconf_exact'},
 {'input_image': '/home/viplab/liukejia/project/result/render_3d_stitched/num123_-800_150_0_0008.png',
  'target_dir': '/tmp/vggt_num123_-800_150_0_0008_rn4cc1n2',
  'glb_path': '/home/viplab/liukejia/project/result/reconstruct_pred_scale2/num123_-800_150_0_0008.glb',
  'npz_path': None,
  'depth_outputs': None,
  'camera_source': 'camconf_exact'},
 {'input_image': '/home/viplab/liukejia/project/result/render_3d_stitched/num123_-800_200_0_0009.png',
  'target_dir': '/tmp/vggt_num123_-800_200_0_0009_cycg0_7i',
  'glb_path': '/home/viplab/liukejia/project/result/reconstruct_pred_scale2/num123_-800_200_0_0009.glb',
  'npz_path': None,
  'depth_

In [13]:
# 快速单图验证（可选）
_test_img = collect_input_images(input_path, valid_exts=valid_exts)[0]
print('[quick-check] image =', _test_img.name)

_quick = reconstruct_one_image(
    image_path=_test_img,
    output_dir=Path(output_path),
    conf_thres=conf_thres,
    prediction_mode=prediction_mode,
    force_resize=force_resize,
    save_npz=False,
    save_raw_depth_npy=save_raw_depth_npy,
    depth_output_dir=depth_output_path,
    only_output_depth=only_output_depth,
    use_camconf_camera=use_camconf_camera,
    camconf_dir=camconf_dir,
    camconf_angle_order=camconf_angle_order,
    strict_camconf=strict_camconf,
    invert_camera_direction=invert_camera_direction,
    post_align_mode=post_align_mode,
    post_align_scale=post_align_scale,
    camera_fit_match_range=camera_fit_match_range,
    camera_fit_range_gain=camera_fit_range_gain,
    camera_fit_scale_min=camera_fit_scale_min,
    camera_fit_scale_max=camera_fit_scale_max,
    camera_fit_range_apply_to_camera=camera_fit_range_apply_to_camera,
 )
print('[quick-check] glb =', _quick['glb_path'] if _quick['glb_path'] else 'SKIPPED (only_output_depth=True)')
if _quick.get('depth_outputs', None) is not None:
    print('[quick-check] depth_npy =', _quick['depth_outputs'].get('depth_npy', None))

[quick-check] image = num123_-800_100_0_0007.png
[DIAG] pts shape=(1073296, 3)
[DIAG] pts min=[-0.28479123 -0.26024848  0.8540226 ], max=[0.26843187 0.23174386 1.1541613 ], mean=[ 0.00103092 -0.03648996  0.9849603 ]
[DIAG] pts span (p95-p5)=0.5608378052711487
[DIAG] cameras centers: [[-9.3604402e-05 -4.6534795e-05  3.0325432e-04]
 [ 4.9613926e-01 -4.3133426e-01  2.5332913e-01]
 [-6.4753378e-03 -8.9428872e-01  5.1086628e-01]
 [-5.1288813e-01 -4.4012478e-01  2.4273330e-01]]
[camera-exact] num123_-800_100_0_0007.png | used_angles=[0, 90, 180, 270] | camera_rms=0.000000 (should be ~0)
[post-align-exact] mode=camera_fit, used_angles=[0, 90, 180, 270], scale=0.8455, camera_locked=True (intr/extr unchanged)
Building GLB scene
Using Depthmap and Camera Branch
GLB Scene built
[quick-check] glb = /home/viplab/liukejia/project/result/reconstruct_pred_scale2/num123_-800_100_0_0007.glb


In [14]:
# 旋转误差检查（raw / anchor-rigid / exact）
import numpy as np
from pathlib import Path


def _to_extr44_batch(extrinsic: np.ndarray) -> np.ndarray:
    ext = np.asarray(extrinsic, dtype=np.float32)
    if ext.ndim != 3:
        raise ValueError(f'extrinsic ndim error: {ext.shape}')
    if ext.shape[1:] == (4, 4):
        return ext
    if ext.shape[1:] == (3, 4):
        out = np.tile(np.eye(4, dtype=np.float32), (ext.shape[0], 1, 1))
        out[:, :3, :4] = ext
        return out
    raise ValueError(f'unsupported extrinsic shape: {ext.shape}')


def _c2w_rot_centers(extrinsic: np.ndarray):
    ext44 = _to_extr44_batch(extrinsic)
    c2w = np.linalg.inv(ext44)
    R = c2w[:, :3, :3]
    C = c2w[:, :3, 3]
    return R.astype(np.float32), C.astype(np.float32)


def _rot_err_deg(Ra: np.ndarray, Rb: np.ndarray) -> np.ndarray:
    # geodesic angle: acos((trace(Ra Rb^T)-1)/2)
    rel = np.einsum('nij,njk->nik', Ra, np.transpose(Rb, (0, 2, 1)))
    tr = rel[:, 0, 0] + rel[:, 1, 1] + rel[:, 2, 2]
    cosv = np.clip((tr - 1.0) / 2.0, -1.0, 1.0)
    return np.degrees(np.arccos(cosv)).astype(np.float32)


def _load_camconf_extr_for_image(image_path: Path, camconf_dir: str, angle_order=(0, 90, 180, 270), H=256, W=256):
    base_stem = extract_camconf_base_stem(image_path.stem)
    exts, used_angles = [], []
    for a in angle_order:
        p = Path(camconf_dir) / f'{base_stem}_{int(a)}_camconf.json'
        if not p.exists():
            continue
        ext44, _ = parse_json_camera_extrinsic_intrinsic(
            str(p), device='cpu', H=H, W=W, dtype=torch.float32, flip_y=True
        )
        exts.append(ext44.squeeze(0).detach().cpu().numpy().astype(np.float32))
        used_angles.append(int(a))
    if len(exts) == 0:
        raise FileNotFoundError(f'no camconf for {image_path.name}')
    return np.stack(exts, axis=0), used_angles


# 选择一张图做检查（默认第一张）
_check_img = collect_input_images(input_path, valid_exts=valid_exts)[0]
print('[rot-check] image =', _check_img.name)

# 复用已有切图流程
_check_rgb = read_single_image_512(str(_check_img), force_resize=force_resize)
_check_dir = Path(tempfile.mkdtemp(prefix=f'vggt_rotchk_{_check_img.stem}_'))
_ = split_512_to_4x256_and_save(_check_rgb, _check_dir)

# 1) raw VGGT（不使用 camconf）
_base_fn = globals().get('_base_run_model_exact', run_model)
pred_raw = _base_fn(
    target_dir=_check_dir,
    model=model,
    image_path=_check_img,
    use_camconf_camera=False,
    camconf_dir=camconf_dir,
    camconf_angle_order=camconf_angle_order,
    strict_camconf=strict_camconf,
    invert_camera_direction=invert_camera_direction,
)

# 2) 旧式 anchor-rigid（用于对照）
pred_anchor = {
    k: (v.copy() if isinstance(v, np.ndarray) else v)
    for k, v in pred_raw.items()
}
H = int(pred_raw['depth'].shape[1])
W = int(pred_raw['depth'].shape[2])
n_pred = int(pred_raw['depth'].shape[0])
anchor_ext44, anchor_intr33, _ = load_camconf_anchor_camera_for_image(
    image_path=_check_img,
    camconf_dir=camconf_dir,
    anchor_angle=int(camconf_angle_order[0]),
    H=H,
    W=W,
)
ext_anchor = _rigid_align_all_cameras_by_cam0(np.asarray(pred_raw['extrinsic'], dtype=np.float32), anchor_ext44)
pred_anchor['extrinsic'] = ext_anchor.astype(np.float32)
pred_anchor['intrinsic'] = np.repeat(anchor_intr33[None, ...], n_pred, axis=0).astype(np.float32)

# 3) 当前 exact 模式输出（camconf_exact）
pred_exact = run_model(
    target_dir=_check_dir,
    model=model,
    image_path=_check_img,
    use_camconf_camera=True,
    camconf_dir=camconf_dir,
    camconf_angle_order=camconf_angle_order,
    strict_camconf=strict_camconf,
    invert_camera_direction=invert_camera_direction,
)

# 文件相机（GT）
file_ext, used_angles = _load_camconf_extr_for_image(
    _check_img, camconf_dir, angle_order=camconf_angle_order, H=H, W=W
)
file_ext = file_ext[:n_pred]

# 统一形状到 4x4 后计算旋转误差和中心误差
R_file, C_file = _c2w_rot_centers(file_ext)

def _report(name: str, ext_any: np.ndarray):
    ext_use = _convert_extrinsic_shape_like(np.asarray(ext_any, dtype=np.float32), file_ext)
    R_pred, C_pred = _c2w_rot_centers(ext_use)
    rot = _rot_err_deg(R_pred, R_file)
    ctr = np.linalg.norm(C_pred - C_file, axis=1)
    print(f'\n[rot-check] {name} | used_angles={used_angles[:len(rot)]}')
    print(f'  rot_err_deg: mean={float(rot.mean()):.4f}, median={float(np.median(rot)):.4f}, max={float(rot.max()):.4f}')
    print(f'  center_err : mean={float(ctr.mean()):.4f}, median={float(np.median(ctr)):.4f}, max={float(ctr.max()):.4f}')

_report('raw_vggt', pred_raw['extrinsic'])
_report('anchor_rigid', pred_anchor['extrinsic'])
_report('exact_camconf', pred_exact['extrinsic'])

print('\n[rot-check] 说明：exact_camconf 的 center_err 应接近 0；若 rot_err 仍偏大，通常是坐标系/朝向定义差异。')

[rot-check] image = num123_-800_100_0_0007.png
[camera-debug] 相机前向(-Z)与点云中心夹角(前4):
  point_center=(0.0010, -0.0365, 0.9850)
  cam[0] forward=(0.0001, -0.0002, -1.0000) to_center=(0.0011, -0.0364, 0.9847) angle_deg=177.87
  cam[1] forward=(0.4857, -0.4183, -0.7676) to_center=(-0.4951, 0.3948, 0.7316) angle_deg=178.27
  cam[2] forward=(-0.0084, -0.8580, -0.5136) to_center=(0.0075, 0.8578, 0.4741) angle_deg=178.02
  cam[3] forward=(-0.4926, -0.4181, -0.7633) to_center=(0.5139, 0.4036, 0.7422) angle_deg=178.19
[DIAG] pts shape=(1073296, 3)
[DIAG] pts min=[-0.28479123 -0.26024848  0.8540226 ], max=[0.26843187 0.23174386 1.1541613 ], mean=[ 0.00103092 -0.03648996  0.9849603 ]
[DIAG] pts span (p95-p5)=0.5608378052711487
[DIAG] cameras centers: [[-9.3604402e-05 -4.6534795e-05  3.0325432e-04]
 [ 4.9613926e-01 -4.3133426e-01  2.5332913e-01]
 [-6.4753378e-03 -8.9428872e-01  5.1086628e-01]
 [-5.1288813e-01 -4.4012478e-01  2.4273330e-01]]
[camera-debug] 相机前向(-Z)与点云中心夹角(前4):
  point_center=(0.0010, 

In [15]:
# 前向轴检查与修正（-Z / +Z）
import numpy as np


def _forward_vecs_with_axis(extrinsic: np.ndarray, axis: str = '-Z') -> np.ndarray:
    ext = np.asarray(extrinsic, dtype=np.float32)
    ext4 = _convert_extrinsic_shape_like(ext, np.zeros((ext.shape[0], 4, 4), dtype=np.float32))
    c2w = np.linalg.inv(ext4)
    if axis.upper() == '+Z':
        fwd = c2w[:, :3, 2]
    else:
        fwd = -c2w[:, :3, 2]
    norm = np.linalg.norm(fwd, axis=1, keepdims=True) + 1e-8
    return (fwd / norm).astype(np.float32)


def _angles_to_point_center(extrinsic: np.ndarray, world_points: np.ndarray, axis: str) -> np.ndarray:
    centers = _camera_centers_from_extr(extrinsic)
    fwd = _forward_vecs_with_axis(extrinsic, axis=axis)
    pts = np.asarray(world_points, dtype=np.float32).reshape(-1, 3)
    point_center = pts.mean(axis=0)
    to_center = point_center[None, :] - centers
    to_center = to_center / (np.linalg.norm(to_center, axis=1, keepdims=True) + 1e-8)
    cosv = np.clip(np.sum(fwd * to_center, axis=1), -1.0, 1.0)
    return np.degrees(np.arccos(cosv)).astype(np.float32)


# 优先用现成变量；没有则快速重建一次
if 'pred_exact' in globals() and isinstance(pred_exact, dict):
    _pred_for_axis = pred_exact
else:
    _img = collect_input_images(input_path, valid_exts=valid_exts)[0]
    _tmp = Path(tempfile.mkdtemp(prefix=f'vggt_axis_{_img.stem}_'))
    _rgb = read_single_image_512(str(_img), force_resize=force_resize)
    _ = split_512_to_4x256_and_save(_rgb, _tmp)
    _pred_for_axis = run_model(
        target_dir=_tmp,
        model=model,
        image_path=_img,
        use_camconf_camera=use_camconf_camera,
        camconf_dir=camconf_dir,
        camconf_angle_order=camconf_angle_order,
        strict_camconf=strict_camconf,
        invert_camera_direction=invert_camera_direction,
    )

_a_neg = _angles_to_point_center(
    _pred_for_axis['extrinsic'],
    _pred_for_axis['world_points_from_depth'],
    axis='-Z',
)
_a_pos = _angles_to_point_center(
    _pred_for_axis['extrinsic'],
    _pred_for_axis['world_points_from_depth'],
    axis='+Z',
)

print('[forward-axis-check] -Z mean/min/max =', float(_a_neg.mean()), float(_a_neg.min()), float(_a_neg.max()))
print('[forward-axis-check] +Z mean/min/max =', float(_a_pos.mean()), float(_a_pos.min()), float(_a_pos.max()))

CAMERA_FORWARD_AXIS = '+Z' if float(_a_pos.mean()) < float(_a_neg.mean()) else '-Z'
print('[forward-axis-check] 推荐前向轴 =', CAMERA_FORWARD_AXIS)

# 覆盖调试函数：让后续 camera-debug / camera-help 使用推荐前向轴
def _camera_forward_vecs_from_extr(extrinsic: np.ndarray) -> np.ndarray:
    return _forward_vecs_with_axis(extrinsic, axis=CAMERA_FORWARD_AXIS)


def _camera_forward_angle_to_point_center(extrinsic: np.ndarray, world_points: np.ndarray):
    centers = _camera_centers_from_extr(extrinsic)
    forward = _camera_forward_vecs_from_extr(extrinsic)
    pts = np.asarray(world_points, dtype=np.float32).reshape(-1, 3)
    point_center = pts.mean(axis=0)
    to_center = point_center[None, :] - centers
    to_center_norm = np.linalg.norm(to_center, axis=1, keepdims=True) + 1e-8
    to_center_unit = to_center / to_center_norm
    cosv = np.sum(forward * to_center_unit, axis=1)
    cosv = np.clip(cosv, -1.0, 1.0)
    angles = np.degrees(np.arccos(cosv)).astype(np.float32)
    return point_center.astype(np.float32), forward.astype(np.float32), to_center.astype(np.float32), angles

print('[forward-axis-check] 已更新调试函数，后续日志按', CAMERA_FORWARD_AXIS, '解释前向')

[forward-axis-check] -Z mean/min/max = 61.242645263671875 61.228816986083984 61.25486755371094
[forward-axis-check] +Z mean/min/max = 118.7573471069336 118.74512481689453 118.77117919921875
[forward-axis-check] 推荐前向轴 = -Z
[forward-axis-check] 已更新调试函数，后续日志按 -Z 解释前向


In [16]:
# 验证前：重载 visual_util，确保使用最新相机可视化修复
import importlib
import visual_util as _vu

importlib.reload(_vu)
predictions_to_glb = _vu.predictions_to_glb
print('[verify] reloaded visual_util from:', _vu.__file__)

[verify] reloaded visual_util from: /home/viplab/liukejia/project/vggt/visual_util.py


In [17]:
# 修复：camconf_exact 下保留稠密点云（不再用“新相机+旧深度”重新反投影）
# 思路：先用原始 VGGT 相机得到点云，再通过相机中心相似变换把点云对齐到 camconf 坐标系；最后仅替换相机为 camconf。

if 'run_model' not in globals():
    print('[EXACT_FIX] 请先运行前面的函数单元')
else:
    _run_model_before_dense_fix = run_model

    def run_model(
        target_dir: str,
        model,
        image_path: Path = None,
        use_camconf_camera: bool = False,
        camconf_dir: str = None,
        camconf_angle_order=(0, 90, 180, 270),
        strict_camconf: bool = True,
        invert_camera_direction: bool = False,
    ) -> dict:
        # 先拿“原始预测点云 + 原始预测相机”
        predictions = _run_model_before_dense_fix(
            target_dir=target_dir,
            model=model,
            image_path=image_path,
            use_camconf_camera=False,
            camconf_dir=camconf_dir,
            camconf_angle_order=camconf_angle_order,
            strict_camconf=strict_camconf,
            invert_camera_direction=invert_camera_direction,
        )

        if not use_camconf_camera:
            return predictions

        if image_path is None or camconf_dir is None:
            raise ValueError('camconf 模式需要 image_path 与 camconf_dir')

        try:
            H, W = int(predictions['depth'].shape[1]), int(predictions['depth'].shape[2])
            n_pred = int(predictions['depth'].shape[0])

            file_ext, file_intr, file_jsons, valid_angles = _load_camconf_all_cameras_for_image(
                image_path=image_path,
                camconf_dir=camconf_dir,
                angle_order=camconf_angle_order,
                H=H,
                W=W,
            )
            if file_ext.shape[0] < n_pred:
                raise ValueError(f'camconf views not enough: need={n_pred}, got={file_ext.shape[0]}')

            # 保存原始预测相机（用于估计相似变换）
            pred_extr_orig = np.asarray(predictions['extrinsic'], dtype=np.float32)
            pred_centers = _camera_centers_from_extr(pred_extr_orig)

            # 目标相机（camconf）形状适配到导出期望
            file_ext_like = _convert_extrinsic_shape_like(file_ext[:n_pred], pred_extr_orig)
            file_centers = _camera_centers_from_extr(np.asarray(file_ext_like, dtype=np.float32))

            # 用相机中心估计相似变换：把“原始点云”搬到 camconf 坐标系
            n = min(len(pred_centers), len(file_centers))
            s, R, t = _estimate_similarity_umeyama(pred_centers[:n], file_centers[:n], with_scale=True)
            predictions = _apply_similarity_to_points_only(predictions, s=s, R=R, t=t)

            # 相机直接锁定为 camconf
            predictions['extrinsic'] = file_ext_like.astype(np.float32)
            predictions['intrinsic'] = file_intr[:n_pred].astype(np.float32)
            predictions['camera_source'] = 'camconf_exact_dense'
            predictions['camera_json_paths'] = np.array(file_jsons[:n_pred], dtype=object)

            # 仅输出统计（不再重投影）
            cam_rms = float(np.sqrt(np.mean(np.sum((pred_centers[:n] - file_centers[:n]) ** 2, axis=1))))
            pts = np.asarray(predictions['world_points_from_depth'], dtype=np.float32).reshape(-1, 3)
            conf = np.asarray(predictions.get('depth_conf', np.ones((pts.shape[0],), dtype=np.float32))).reshape(-1)
            thr = np.percentile(conf, conf_thres) if conf.size else 0.0
            kept = int(np.sum((conf >= thr) & (conf > 1e-5))) if conf.size else int(pts.shape[0])
            print(
                f'[EXACT_FIX] {image_path.name} | mode=camconf_exact_dense | '
                f'cam_rms(before_swap)={cam_rms:.4f} | similarity_scale={float(s):.4f} | points_kept@p{conf_thres}={kept}'
            )

        except Exception as e:
            if strict_camconf:
                raise
            print(f'[Warn][EXACT_FIX] failed, fallback prediction: {e}')

        return predictions

    print('[EXACT_FIX] enabled: keep dense points + lock cameras to camconf')

[EXACT_FIX] enabled: keep dense points + lock cameras to camconf


In [18]:
# 修复补丁：camconf_exact_dense 时后处理只改点云，不改相机
if 'apply_global_rigid_similarity' in globals():
    _apply_before_dense_lock = apply_global_rigid_similarity

    def apply_global_rigid_similarity(
        predictions: dict,
        mode: str = 'rigid',
        scale: float = 1.0,
        image_path: Path = None,
        camconf_dir: str = None,
        camconf_angle_order=(0, 90, 180, 270),
        camera_fit_match_range: bool = True,
        camera_fit_range_gain: float = 1.0,
        camera_fit_scale_min: float = 0.25,
        camera_fit_scale_max: float = 4.0,
        camera_fit_range_apply_to_camera: bool = True,
    ) -> dict:
        src = predictions.get('camera_source', '')
        if src in ('camconf_exact', 'camconf_exact_dense'):
            mode_l = (mode or 'none').lower()
            if mode_l == 'none':
                return predictions
            if image_path is None or camconf_dir is None:
                return predictions

            H, W = int(predictions['depth'].shape[1]), int(predictions['depth'].shape[2])
            file_centers, _ = _load_file_camera_centers_for_image(
                image_path=image_path,
                camconf_dir=camconf_dir,
                angle_order=camconf_angle_order,
                H=H,
                W=W,
            )
            wp = np.asarray(predictions['world_points_from_depth'], dtype=np.float32)
            pts = wp.reshape(-1, 3)
            pts_center = pts.mean(axis=0)
            tgt_xy = file_centers[:, :2].mean(axis=0)
            target_center = np.array([tgt_xy[0], tgt_xy[1], 0.0], dtype=np.float32)

            if mode_l == 'rigid':
                s = 1.0
            elif mode_l == 'similarity':
                s = float(scale) if float(scale) > 0 else 1.0
            else:
                pred_span = _robust_span_diag(pts, q_low=5.0, q_high=95.0)
                file_span = _robust_span_diag(file_centers, q_low=5.0, q_high=95.0)
                if camera_fit_match_range and pred_span > 1e-6 and file_span > 1e-6:
                    raw_s = file_span / pred_span
                    mixed_s = 1.0 + float(camera_fit_range_gain) * (raw_s - 1.0)
                    s = float(np.clip(mixed_s, camera_fit_scale_min, camera_fit_scale_max))
                else:
                    s = 1.0

            t = (target_center - s * pts_center).astype(np.float32)
            predictions = _apply_similarity_to_points_only(predictions, s=s, R=np.eye(3, dtype=np.float32), t=t)
            print(f'[EXACT_FIX] post-align points-only | mode={mode_l}, scale={s:.4f}, camera_locked=True')
            return predictions

        return _apply_before_dense_lock(
            predictions=predictions,
            mode=mode,
            scale=scale,
            image_path=image_path,
            camconf_dir=camconf_dir,
            camconf_angle_order=camconf_angle_order,
            camera_fit_match_range=camera_fit_match_range,
            camera_fit_range_gain=camera_fit_range_gain,
            camera_fit_scale_min=camera_fit_scale_min,
            camera_fit_scale_max=camera_fit_scale_max,
            camera_fit_range_apply_to_camera=camera_fit_range_apply_to_camera,
        )

    print('[EXACT_FIX] apply_global patched: camera stays locked for camconf_exact_dense')

[EXACT_FIX] apply_global patched: camera stays locked for camconf_exact_dense


In [19]:
# 计算当前导出 GLB 的点云中心坐标（对 input_path 下每个样本）
import numpy as np
import pandas as pd
import trimesh
from pathlib import Path

images = collect_input_images(input_path, valid_exts=valid_exts)
out_dir = Path(output_path)
rows = []
missing = []

for img in images:
    glb_path = out_dir / f"{img.stem}.glb"
    if not glb_path.exists():
        missing.append(str(glb_path))
        continue

    scene_or_mesh = trimesh.load(str(glb_path), force='scene')
    pts_list = []
    if isinstance(scene_or_mesh, trimesh.Scene):
        for g in scene_or_mesh.geometry.values():
            if hasattr(g, 'vertices'):
                v = np.asarray(g.vertices)
                if v.ndim == 2 and v.shape[1] == 3 and len(v) > 0:
                    pts_list.append(v)
    else:
        v = np.asarray(scene_or_mesh.vertices)
        if v.ndim == 2 and v.shape[1] == 3 and len(v) > 0:
            pts_list.append(v)

    if len(pts_list) == 0:
        missing.append(str(glb_path) + ' (no vertices)')
        continue

    pts = np.concatenate(pts_list, axis=0)
    center_mean = pts.mean(axis=0)
    center_median = np.median(pts, axis=0)

    rows.append({
        'name': img.stem,
        'glb_path': str(glb_path),
        'point_count': int(pts.shape[0]),
        'mean_x': float(center_mean[0]),
        'mean_y': float(center_mean[1]),
        'mean_z': float(center_mean[2]),
        'median_x': float(center_median[0]),
        'median_y': float(center_median[1]),
        'median_z': float(center_median[2]),
    })

if len(rows) == 0:
    raise RuntimeError('没有可用的 GLB 可计算中心点，请先运行重建单元。')

df_centers = pd.DataFrame(rows).sort_values('name').reset_index(drop=True)
out_csv = out_dir / 'point_centers_all.csv'
df_centers.to_csv(out_csv, index=False)

print('[center-all] images_total =', len(images))
print('[center-all] glb_used =', len(df_centers))
print('[center-all] csv =', out_csv)
if len(missing):
    print('[center-all] missing_or_invalid =', len(missing))
    print('  first 5:')
    for x in missing[:5]:
        print('   -', x)

df_centers.head(10)

[center-all] images_total = 9
[center-all] glb_used = 9
[center-all] csv = /home/viplab/liukejia/project/result/reconstruct_pred_scale2/point_centers_all.csv


,name,glb_path,point_count,mean_x,mean_y,mean_z,median_x,median_y,median_z
0,num123_-800_100_0_0007,/home/viplab/liukejia/project/result/reconstru...,858693,-787.795141,97.484612,-1.114311,-800.848328,99.986374,-1.132629
1,num123_-800_150_0_0008,/home/viplab/liukejia/project/result/reconstru...,858694,-788.632850,151.439815,-1.107793,-800.845947,150.659515,-1.126572
2,num123_-800_200_0_0009,/home/viplab/liukejia/project/result/reconstru...,858693,-785.134245,200.813736,-1.114008,-800.833252,200.125381,-1.129135
3,num123_-850_100_0_0004,/home/viplab/liukejia/project/result/reconstru...,858693,-837.841969,99.220137,-1.109406,-849.494751,99.994995,-1.125870
4,num123_-850_150_0_0005,/home/viplab/liukejia/project/result/reconstru...,858693,-836.696750,149.798777,-1.115929,-849.488098,150.642487,-1.146378
5,num123_-850_200_0_0006,/home/viplab/liukejia/project/result/reconstru...,858693,-835.907164,201.873409,-1.125142,-849.477905,200.133545,-1.146866
6,num123_-900_100_0_0001,/home/viplab/liukejia/project/result/reconstru...,858693,-886.946603,97.132661,-1.121043,-898.652100,99.983070,-1.138809
7,num123_-900_150_0_0002,/home/viplab/liukejia/project/result/reconstru...,858693,-889.522505,157.651236,-1.132745,-898.671021,150.680389,-1.166016
8,num123_-900_200_0_0003,/home/viplab/liukejia/project/result/reconstru...,858693,-885.374088,203.976273,-1.111855,-898.661682,200.147903,-1.130554


## 8.1) 复现检查与差异定位（环境与函数快照）

在复现前后各跑一次，确保函数版本/导入路径一致。

In [20]:
import inspect
import hashlib
import sys
import os
import importlib
from pathlib import Path

def _hash_src(fn):
    try:
        src = inspect.getsource(fn)
    except Exception:
        return 'N/A'
    return hashlib.md5(src.encode('utf-8')).hexdigest()

def _print_snapshot():
    print('=== SNAPSHOT ===')
    print('cwd =', os.getcwd())
    print('PROJECT_ROOT =', PROJECT_ROOT)
    print('input_path =', input_path)
    print('output_path =', output_path)
    print('camconf_dir =', camconf_dir)
    print('use_camconf_camera =', use_camconf_camera)
    print('post_align_mode =', post_align_mode)
    print('camera_fit_range_apply_to_camera =', camera_fit_range_apply_to_camera)
    # visual_util 路径
    try:
        import visual_util as _vu
        print('visual_util =', _vu.__file__)
    except Exception as e:
        print('visual_util import failed:', e)
    # 关键函数 hash
    print('run_model hash =', _hash_src(run_model))
    print('apply_global_rigid_similarity hash =', _hash_src(apply_global_rigid_similarity))
    print('predictions_to_glb hash =', _hash_src(predictions_to_glb))
    # patch flags
    print('EXACT_CAMCONF_MODE =', globals().get('EXACT_CAMCONF_MODE', None))
    print('EXACT_POINT_SCALE_TO_CAMCONF =', globals().get('EXACT_POINT_SCALE_TO_CAMCONF', None))
    print('')

_print_snapshot()

=== SNAPSHOT ===
cwd = /home/viplab/liukejia/project/vggt
PROJECT_ROOT = /home/viplab/liukejia/project/vggt
input_path = /home/viplab/liukejia/project/result/render_3d_stitched/
output_path = /home/viplab/liukejia/project/result/reconstruct_pred_scale2/
camconf_dir = /data/test_grid4_from_test_grid4/camconf
use_camconf_camera = True
post_align_mode = camera_fit
camera_fit_range_apply_to_camera = True
visual_util = /home/viplab/liukejia/project/vggt/visual_util.py
run_model hash = e4608f08acf167117360db05e3f7432a
apply_global_rigid_similarity hash = 6b30dc870889001ba38f4b813c47a253
predictions_to_glb hash = 307b4a47c542ebe4bc39adce70dbbf96
EXACT_CAMCONF_MODE = True
EXACT_POINT_SCALE_TO_CAMCONF = True



## 8.2) 结果差异快速对比（CSV/GLB）

对比两个输出目录的中心点 CSV；若 CSV 不存在会提示。

In [21]:
import pandas as pd
import numpy as np
import trimesh

old_dir = '/home/viplab/liukejia/project/result/reconstruct_gt_scale'
new_dir = '/home/viplab/liukejia/project/result/reconstruct_gt_scale2'

csv_old = Path(old_dir) / 'point_centers_all.csv'
csv_new = Path(new_dir) / 'point_centers_all.csv'

if not csv_old.exists() or not csv_new.exists():
    print('[diff] csv missing:', csv_old, csv_new)
else:
    df_old = pd.read_csv(csv_old)
    df_new = pd.read_csv(csv_new)
    m = df_old.merge(df_new, on='name', suffixes=('_old','_new'))
    for col in ['mean_x','mean_y','mean_z','median_x','median_y','median_z']:
        m[col + '_diff'] = m[col + '_new'] - m[col + '_old']
    m['abs_max_diff'] = m[[c for c in m.columns if c.endswith('_diff')]].abs().max(axis=1)
    print('[diff] merged rows =', len(m))
    print(m[['abs_max_diff']].describe())
    print('[diff] top 5:')
    print(m.sort_values('abs_max_diff', ascending=False).head(5)[['name','abs_max_diff']])

# 可选：对单个样本比较 GLB 点云中心
sample_name = None  # 例如 'num123_-900_200_0_0003'
if sample_name:
    p_old = Path(old_dir) / f'{sample_name}.glb'
    p_new = Path(new_dir) / f'{sample_name}.glb'
    if not p_old.exists() or not p_new.exists():
        print('[glb] missing:', p_old, p_new)
    else:
        def _load_pts(p):
            scene = trimesh.load(str(p), force='scene')
            pts_list = []
            if isinstance(scene, trimesh.Scene):
                for g in scene.geometry.values():
                    if hasattr(g, 'vertices'):
                        v = np.asarray(g.vertices)
                        if v.ndim == 2 and v.shape[1] == 3 and len(v) > 0:
                            pts_list.append(v)
            else:
                v = np.asarray(scene.vertices)
                if v.ndim == 2 and v.shape[1] == 3 and len(v) > 0:
                    pts_list.append(v)
            return np.concatenate(pts_list, axis=0)
        pts_old = _load_pts(p_old)
        pts_new = _load_pts(p_new)
        c_old = pts_old.mean(axis=0)
        c_new = pts_new.mean(axis=0)
        print('[glb] center old =', c_old)
        print('[glb] center new =', c_new)
        print('[glb] center diff =', c_new - c_old)

[diff] csv missing: /home/viplab/liukejia/project/result/reconstruct_gt_scale/point_centers_all.csv /home/viplab/liukejia/project/result/reconstruct_gt_scale2/point_centers_all.csv


In [22]:
import numpy as np
import trimesh
from pathlib import Path

sample_glb = Path('/home/viplab/liukejia/project/result/reconstruct_gt_scale2/num123_-800_100_0_0007.glb')
print('glb =', sample_glb)

if not sample_glb.exists():
    print('[glb-check] missing file')
else:
    scene = trimesh.load(str(sample_glb), force='scene')
    geom_count = len(scene.geometry) if isinstance(scene, trimesh.Scene) else 1
    pts_list = []
    if isinstance(scene, trimesh.Scene):
        for name, g in scene.geometry.items():
            if hasattr(g, 'vertices'):
                v = np.asarray(g.vertices)
                if v.ndim == 2 and v.shape[1] == 3 and len(v) > 0:
                    pts_list.append(v)
                    print(f'[geom] {name}: pts={len(v)} bbox_min={v.min(axis=0)} bbox_max={v.max(axis=0)}')
            else:
                print(f'[geom] {name}: no vertices')
    else:
        v = np.asarray(scene.vertices)
        if v.ndim == 2 and v.shape[1] == 3 and len(v) > 0:
            pts_list.append(v)
            print(f'[geom] mesh: pts={len(v)} bbox_min={v.min(axis=0)} bbox_max={v.max(axis=0)}')

    if len(pts_list) == 0:
        print('[glb-check] no vertices found')
    else:
        pts = np.concatenate(pts_list, axis=0)
        center = pts.mean(axis=0)
        span = np.linalg.norm(np.percentile(pts, 95, axis=0) - np.percentile(pts, 5, axis=0))
        print('[glb-check] total_pts =', len(pts))
        print('[glb-check] center =', center)
        print('[glb-check] span(p5-p95) =', span)
        print('[glb-check] geom_count =', geom_count)

glb = /home/viplab/liukejia/project/result/reconstruct_gt_scale2/num123_-800_100_0_0007.glb
[glb-check] missing file


In [23]:
import numpy as np
import trimesh
from pathlib import Path

def _load_pts_from_glb(p: Path):
    scene = trimesh.load(str(p), force='scene')
    pts_list = []
    if isinstance(scene, trimesh.Scene):
        for g in scene.geometry.values():
            if hasattr(g, 'vertices'):
                v = np.asarray(g.vertices)
                if v.ndim == 2 and v.shape[1] == 3 and len(v) > 0:
                    pts_list.append(v)
    else:
        v = np.asarray(scene.vertices)
        if v.ndim == 2 and v.shape[1] == 3 and len(v) > 0:
            pts_list.append(v)
    if len(pts_list) == 0:
        return None
    return np.concatenate(pts_list, axis=0)

def _summarize_pts(pts: np.ndarray):
    center = pts.mean(axis=0)
    bbox_min = pts.min(axis=0)
    bbox_max = pts.max(axis=0)
    span = np.linalg.norm(np.percentile(pts, 95, axis=0) - np.percentile(pts, 5, axis=0))
    thick = bbox_max - bbox_min
    return {
        'count': int(len(pts)),
        'center': center,
        'bbox_min': bbox_min,
        'bbox_max': bbox_max,
        'span_p5p95': float(span),
        'thickness_xyz': thick,
    }

sample_name = 'num123_-800_100_0_0007'
old_glb = Path('/home/viplab/liukejia/project/result/reconstruct_gt_scale') / f'{sample_name}.glb'
new_glb = Path('/home/viplab/liukejia/project/result/reconstruct_gt_scale2') / f'{sample_name}.glb'

for tag, p in [('old', old_glb), ('new', new_glb)]:
    print(f'[{tag}] {p}')
    if not p.exists():
        print('  missing')
        continue
    pts = _load_pts_from_glb(p)
    if pts is None:
        print('  no vertices')
        continue
    s = _summarize_pts(pts)
    print('  count =', s['count'])
    print('  center =', s['center'])
    print('  bbox_min =', s['bbox_min'])
    print('  bbox_max =', s['bbox_max'])
    print('  thickness_xyz =', s['thickness_xyz'])
    print('  span_p5p95 =', s['span_p5p95'])

[old] /home/viplab/liukejia/project/result/reconstruct_gt_scale/num123_-800_100_0_0007.glb
  missing
[new] /home/viplab/liukejia/project/result/reconstruct_gt_scale2/num123_-800_100_0_0007.glb
  missing


In [24]:
import numpy as np
import tempfile
from pathlib import Path

sample_name = 'num123_-800_100_0_0007'
img_path = next((p for p in input_images if p.stem == sample_name), None)
if img_path is None:
    img_path = input_images[0]
print('[test] image =', img_path.name)

def _clone_predictions(pred: dict) -> dict:
    out = {}
    for k, v in pred.items():
        if isinstance(v, np.ndarray):
            out[k] = v.copy()
        else:
            out[k] = v
    return out

def _stats_from_points(points: np.ndarray) -> dict:
    pts = np.asarray(points, dtype=np.float32).reshape(-1, 3)
    bbox_min = pts.min(axis=0)
    bbox_max = pts.max(axis=0)
    center = pts.mean(axis=0)
    thickness = bbox_max - bbox_min
    span = np.linalg.norm(np.percentile(pts, 95, axis=0) - np.percentile(pts, 5, axis=0))
    return {
        'count': int(len(pts)),
        'center': center,
        'bbox_min': bbox_min,
        'bbox_max': bbox_max,
        'thickness': thickness,
        'span_p5p95': float(span),
    }

# 准备切图目录
img_rgb = read_single_image_512(str(img_path), force_resize=force_resize)
tmp_dir = Path(tempfile.mkdtemp(prefix=f'vggt_ablation_{img_path.stem}_'))
_ = split_512_to_4x256_and_save(img_rgb, tmp_dir)

# 准备 run_model 变体
variants = []
base_run = globals().get('_base_run_model_exact', None)
exact_run = globals().get('_run_model_before_dense_fix', None)
current_run = run_model

if base_run is not None:
    variants.append(('raw_pred (base_run, camconf=False)', base_run, False))
    variants.append(('anchor_rigid (base_run, camconf=True)', base_run, True))
else:
    variants.append(('raw_pred (current_run, camconf=False)', current_run, False))

if exact_run is not None:
    variants.append(('camconf_exact (exact_run, camconf=True)', exact_run, True))

variants.append(('current_run (camconf=True)', current_run, True))

# 运行并统计
modes = [
    ('none', None),
    ('rigid', None),
    ('camera_fit', True),
    ('camera_fit', False),
]

for vname, run_fn, use_camconf in variants:
    print('\n===', vname, '===')
    pred = run_fn(
        target_dir=tmp_dir,
        model=model,
        image_path=img_path,
        use_camconf_camera=use_camconf,
        camconf_dir=camconf_dir,
        camconf_angle_order=camconf_angle_order,
        strict_camconf=strict_camconf,
        invert_camera_direction=invert_camera_direction,
    )
    if 'world_points_from_depth' not in pred:
        print('[warn] no world_points_from_depth')
        continue

    base_stats = _stats_from_points(pred['world_points_from_depth'])
    print('[base] thickness=', base_stats['thickness'], 'span=', base_stats['span_p5p95'])

    for mode, apply_to_cam in modes:
        p2 = _clone_predictions(pred)
        if mode == 'camera_fit':
            p2 = apply_global_rigid_similarity(
                p2,
                mode=mode,
                scale=post_align_scale,
                image_path=img_path,
                camconf_dir=camconf_dir,
                camconf_angle_order=camconf_angle_order,
                camera_fit_match_range=camera_fit_match_range,
                camera_fit_range_gain=camera_fit_range_gain,
                camera_fit_scale_min=camera_fit_scale_min,
                camera_fit_scale_max=camera_fit_scale_max,
                camera_fit_range_apply_to_camera=bool(apply_to_cam),
            )
            tag = f'{mode}|apply_to_cam={apply_to_cam}'
        elif mode == 'rigid':
            p2 = apply_global_rigid_similarity(
                p2,
                mode=mode,
                scale=post_align_scale,
                image_path=img_path,
                camconf_dir=camconf_dir,
                camconf_angle_order=camconf_angle_order,
            )
            tag = mode
        else:
            tag = mode

        st = _stats_from_points(p2['world_points_from_depth'])
        print(f'  [{tag}] thickness={st["thickness"]} span={st["span_p5p95"]}')

[test] image = num123_-800_100_0_0007.png

=== raw_pred (base_run, camconf=False) ===
[camera-debug] 相机前向(-Z)与点云中心夹角(前4):
  point_center=(0.0010, -0.0365, 0.9850)
  cam[0] forward=(0.0001, -0.0002, -1.0000) to_center=(0.0011, -0.0364, 0.9847) angle_deg=177.87
  cam[1] forward=(0.4857, -0.4183, -0.7676) to_center=(-0.4951, 0.3948, 0.7316) angle_deg=178.27
  cam[2] forward=(-0.0084, -0.8580, -0.5136) to_center=(0.0075, 0.8578, 0.4741) angle_deg=178.02
  cam[3] forward=(-0.4926, -0.4181, -0.7633) to_center=(0.5139, 0.4036, 0.7422) angle_deg=178.19
[DIAG] pts shape=(1073296, 3)
[DIAG] pts min=[-0.28479123 -0.26024848  0.8540226 ], max=[0.26843187 0.23174386 1.1541613 ], mean=[ 0.00103092 -0.03648996  0.9849603 ]
[DIAG] pts span (p95-p5)=0.5608378052711487
[DIAG] cameras centers: [[-9.3604402e-05 -4.6534795e-05  3.0325432e-04]
 [ 4.9613926e-01 -4.3133426e-01  2.5332913e-01]
 [-6.4753378e-03 -8.9428872e-01  5.1086628e-01]
 [-5.1288813e-01 -4.4012478e-01  2.4273330e-01]]
[base] thickness= [0.